In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/spikie/capstone-split/train_tokens.bin
/kaggle/input/datasets/spikie/capstone-split/train_manifest.json
/kaggle/input/datasets/spikie/capstone-split/metadata.json
/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl
/kaggle/input/datasets/spikie/capstone-split/valid_manifest.json
/kaggle/input/datasets/spikie/capstone-split/legal_bpe_64k.pkl
/kaggle/input/datasets/spikie/capstone-split/valid_tokens.bin


In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
#Custom BPE Tokenizer adapted from A-1
from collections import Counter,defaultdict
import heapq
import re
from tqdm.auto import tqdm

class ByteLevelBPE:
    SPECIALS=["<eos>"]

    def __init__(self,vocab_size=32000):
        self.target_vocab_size=vocab_size
        self._merge_id_pairs=[]
        self.merges=[]
        self._token_bytes=[]
        self._bytes_to_id={}
        self._merge_ranks={}
        self._merge_to_new_id={}
        self.itos=[]
        self.stoi={}
        self.vocab_size=0
        self.eos_id=0

    @staticmethod
    def _get_word_pairs(symbols):
        return Counter(zip(symbols,symbols[1:]))

    @staticmethod
    def _merge_word(symbols,pair,new_id):
        left,right=pair
        merged=[]
        i=0

        while i<len(symbols):
            if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                merged.append(new_id)
                i+=2
            else:
                merged.append(symbols[i])
                i+=1

        return merged

    def train_from_word_counts(self,word_freqs):
        self._token_bytes=[(byte,) for byte in range(256)]
        words=[]
        freqs=[]
        for word,freq in tqdm(word_freqs.items(),desc="Preparing BPE corpus"):
            symbols=list(word.encode("utf-8"))
            if len(symbols)>=2:
                words.append(symbols)
                freqs.append(freq)

        pair_counts=Counter()
        pair_to_words=defaultdict(set)
        for word_idx,symbols in tqdm(enumerate(words),total=len(words),desc="Building pair statistics"):
            word_freq=freqs[word_idx]
            for pair,occurrences in self._get_word_pairs(symbols).items():
                pair_counts[pair]+=occurrences*word_freq
                pair_to_words[pair].add(word_idx)

        heap=[]
        for pair,count in pair_counts.items():
            left,right=pair
            heap.append((-count,self._token_bytes[left],self._token_bytes[right],pair))
        heapq.heapify(heap)
        max_merges=self.target_vocab_size-256-len(self.SPECIALS)
        for _ in tqdm(range(max_merges),desc="Learning BPE merges"):
            best_pair=None
            while heap:
                neg_count,_,_,pair=heapq.heappop(heap)
                count=pair_counts.get(pair,0)
                if count==-neg_count and count>1:
                    best_pair=pair
                    break

            if best_pair is None:
                break

            left,right=best_pair
            new_id=len(self._token_bytes)
            self._token_bytes.append(self._token_bytes[left]+self._token_bytes[right])
            self._merge_id_pairs.append(best_pair)
            self.merges.append((self._token_bytes[left],self._token_bytes[right]))
            affected_words=list(pair_to_words.get(best_pair,set()))
            changed_pairs=set()
            for word_idx in affected_words:
                symbols=words[word_idx]
                word_freq=freqs[word_idx]
                new_symbols=[]
                i=0
                while i<len(symbols):
                    if i+1<len(symbols) and symbols[i]==left and symbols[i+1]==right:
                        new_symbols.append(new_id)
                        i+=2
                    else:
                        new_symbols.append(symbols[i])
                        i+=1

                if new_symbols==symbols:
                    continue
                old_pairs=self._get_word_pairs(symbols)
                new_pairs=self._get_word_pairs(new_symbols)
                for pair,occurrences in old_pairs.items():
                    pair_counts[pair]-=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair,occurrences in new_pairs.items():
                    pair_counts[pair]+=occurrences*word_freq
                    changed_pairs.add(pair)

                for pair in old_pairs:
                    if pair not in new_pairs:
                        pair_to_words[pair].discard(word_idx)

                for pair in new_pairs:
                    pair_to_words[pair].add(word_idx)

                words[word_idx]=new_symbols
            pair_counts.pop(best_pair,None)
            pair_to_words.pop(best_pair,None)
            for pair in changed_pairs:
                count=pair_counts.get(pair,0)

                if count<=0:
                    pair_counts.pop(pair,None)
                    pair_to_words.pop(pair,None)
                else:
                    left_pair,right_pair=pair
                    heapq.heappush(heap,(-count,self._token_bytes[left_pair],self._token_bytes[right_pair],pair))

        self._bytes_to_id={token_bytes:idx for idx,token_bytes in enumerate(self._token_bytes)}
        self._merge_ranks={pair:rank for rank,pair in enumerate(self._merge_id_pairs)}
        self._merge_to_new_id={pair:256+rank for rank,pair in enumerate(self._merge_id_pairs)}
        self.itos=self.SPECIALS.copy()
        self.itos.extend(self._token_bytes)
        self.stoi={token:i for i,token in enumerate(self.itos)}
        self.eos_id=0
        self.vocab_size=len(self.itos)
        print("Learned merges:",len(self.merges))
        print("Final vocabulary size:",self.vocab_size)
        return self

    def encode_word(self,word):
        symbols=[byte for byte in word.encode("utf-8")]
        while len(symbols)>=2:
            best_rank=None
            best_pair=None
            for i in range(len(symbols)-1):
                pair=(symbols[i],symbols[i+1])
                rank=self._merge_ranks.get(pair)
                if rank is not None and (best_rank is None or rank<best_rank):
                    best_rank=rank
                    best_pair=pair

            if best_pair is None:
                break

            symbols=self._merge_word(symbols,best_pair,self._merge_to_new_id[best_pair])
        return [token_id+1 for token_id in symbols]

    def encode_text(self,text,add_eos=True):
        tokens=[]

        for part in re.findall(r"\s+|\S+",text):
            tokens.extend(self.encode_word(part))

        if add_eos:
            tokens.append(self.eos_id)

        return tokens

In [6]:
import os,json,pickle,numpy as np,torch

DATA_DIR="/kaggle/input/datasets/spikie/capstone-split"
BPE_PATH=f"{DATA_DIR}/legal_bpe_32k.pkl"
TRAIN_BIN=f"{DATA_DIR}/train_tokens.bin"
VALID_BIN=f"{DATA_DIR}/valid_tokens.bin"

print("DATA_DIR:",DATA_DIR)
print("Train size:",os.path.getsize(TRAIN_BIN)/1e9,"GB")
print("Valid size:",os.path.getsize(VALID_BIN)/1e9,"GB")

with open(f"{DATA_DIR}/metadata.json","r") as f:
    metadata=json.load(f)

print("\nMetadata:")
print(json.dumps(metadata,indent=2))

with open(BPE_PATH,"rb") as f:
    bpe=pickle.load(f)

print("\nBPE loaded")
print("Vocab size:",bpe.vocab_size)
print("EOS id:",bpe.eos_id)

train_tokens=np.memmap(TRAIN_BIN,dtype=np.uint16,mode="r")
valid_tokens=np.memmap(VALID_BIN,dtype=np.uint16,mode="r")

print("\nToken counts:")
print("Train:",len(train_tokens))
print("Valid:",len(valid_tokens))

print("\nFirst 20 train tokens:")
print(train_tokens[:20].tolist())

print("\nFirst 20 valid tokens:")
print(valid_tokens[:20].tolist())

DATA_DIR: /kaggle/input/datasets/spikie/capstone-split
Train size: 11.467042816 GB
Valid size: 1.280978944 GB

Metadata:
{
  "vocab_size": 32000,
  "block_size": 2048,
  "dtype": "uint16",
  "eos_id": 0,
  "train": {
    "documents": 3131637,
    "tokens_written": 5733521408,
    "blocks": 2799571,
    "leftover_tokens": 1624,
    "time_sec": 4212.135697364807,
    "cache_size": 17894465,
    "file_size_bytes": 11467042816
  },
  "valid": {
    "documents": 348608,
    "tokens_written": 640489472,
    "blocks": 312739,
    "leftover_tokens": 1890,
    "time_sec": 1221.6565062999725,
    "cache_size": 19295799,
    "file_size_bytes": 1280978944
  },
  "total_time_sec": 5433.807720422745
}

BPE loaded
Vocab size: 32000
EOS id: 0

Token counts:
Train: 5733521408
Valid: 640489472

First 20 train tokens:
[4970, 33, 6627, 33, 10332, 33, 7266, 33, 10408, 33, 5378, 33, 11018, 2782, 33, 276, 33, 21307, 19773, 33]

First 20 valid tokens:
[323, 33, 4970, 33, 22796, 33, 41, 3601, 33, 2374, 33, 648

In [4]:
import math,torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE=32000
BLOCK_SIZE=2048
D_MODEL=768
N_HEADS=12
N_KV_HEADS=4
HEAD_DIM=64
WINDOW_SIZE=512

assert D_MODEL==N_HEADS*HEAD_DIM
assert N_HEADS%N_KV_HEADS==0

class RotaryEmbedding(nn.Module):
    def __init__(self,head_dim,max_seq_len=2048,base=10000):
        super().__init__()
        inv_freq=1.0/(base**(torch.arange(0,head_dim,2).float()/head_dim))
        pos=torch.arange(max_seq_len).float()
        freqs=torch.outer(pos,inv_freq)
        self.register_buffer("cos",freqs.cos(),persistent=False)
        self.register_buffer("sin",freqs.sin(),persistent=False)

    def forward(self,q,k):
        T=q.size(-2)
        cos=self.cos[:T].to(q.dtype)[None,None,:,:]
        sin=self.sin[:T].to(q.dtype)[None,None,:,:]

        q1=q[...,::2]
        q2=q[...,1::2]
        k1=k[...,::2]
        k2=k[...,1::2]

        q_rot=torch.stack(
            [q1*cos-q2*sin,q1*sin+q2*cos],
            dim=-1
        ).flatten(-2)

        k_rot=torch.stack(
            [k1*cos-k2*sin,k1*sin+k2*cos],
            dim=-1
        ).flatten(-2)

        return q_rot,k_rot

class M3GQAAttention(nn.Module):
    def __init__(
        self,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        head_dim=64,
        window_size=512,
        dropout=0.05
    ):
        super().__init__()

        self.d_model=d_model
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.head_dim=head_dim
        self.window_size=window_size
        self.dropout=dropout

        self.q_proj=nn.Linear(
            d_model,
            n_heads*head_dim,
            bias=False
        )

        self.k_proj=nn.Linear(
            d_model,
            n_kv_heads*head_dim,
            bias=False
        )

        self.v_proj=nn.Linear(
            d_model,
            n_kv_heads*head_dim,
            bias=False
        )

        self.o_proj=nn.Linear(
            n_heads*head_dim,
            d_model,
            bias=False
        )

        self.rope=RotaryEmbedding(
            head_dim,
            BLOCK_SIZE
        )

    def forward(self,x):

        B,T,C=x.shape

        q=self.q_proj(x)
        k=self.k_proj(x)
        v=self.v_proj(x)

        q=q.view(
            B,T,self.n_heads,self.head_dim
        ).transpose(1,2)

        k=k.view(
            B,T,self.n_kv_heads,self.head_dim
        ).transpose(1,2)

        v=v.view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)

        q,k=self.rope(q,k)

        repeat_factor=self.n_heads//self.n_kv_heads

        k=k.repeat_interleave(
            repeat_factor,
            dim=1
        )

        v=v.repeat_interleave(
            repeat_factor,
            dim=1
        )

        # Sliding-window causal attention.
        # Each token attends only to itself and the
        # previous WINDOW_SIZE-1 tokens.
        positions=torch.arange(T,device=x.device)

        mask=(
            (positions[None,:]<=positions[:,None]) &
            (positions[None,:]>=(positions[:,None]-self.window_size+1))
        )

        mask=mask.unsqueeze(0).unsqueeze(0)
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=self.dropout if self.training else 0.0)
        y=y.transpose(1,2).contiguous().view(
            B,T,self.n_heads*self.head_dim
        )

        return self.o_proj(y)

In [5]:
import time,torch

attn=M3GQAAttention().cuda().half().train()

x=torch.randn(
    2,
    2048,
    D_MODEL,
    device="cuda",
    dtype=torch.float16
)

for _ in range(3):
    y=attn(x)

torch.cuda.synchronize()

start=time.time()

for _ in range(10):
    y=attn(x)

torch.cuda.synchronize()

elapsed=time.time()-start

print("10 forwards:",elapsed,"sec")
print("Per forward:",elapsed/10,"sec")
print("Output:",y.shape)

print("\nMemory:")
print("Allocated:",torch.cuda.memory_allocated()/1e9,"GB")
print("Peak:",torch.cuda.max_memory_allocated()/1e9,"GB")

10 forwards: 0.1040639877319336 sec
Per forward: 0.010406398773193359 sec
Output: torch.Size([2, 2048, 768])

Memory:
Allocated: 0.623130624 GB
Peak: 0.669743104 GB


In [6]:
class SwiGLU(nn.Module):
    def __init__(self,d_model=768,ffn_dim=2304):
        super().__init__()

        self.gate_proj=nn.Linear(
            d_model,
            ffn_dim,
            bias=False
        )

        self.up_proj=nn.Linear(
            d_model,
            ffn_dim,
            bias=False
        )

        self.down_proj=nn.Linear(
            ffn_dim,
            d_model,
            bias=False
        )

    def forward(self,x):
        return self.down_proj(
            F.silu(self.gate_proj(x))*self.up_proj(x)
        )

class M3Block(nn.Module):
    def __init__(
        self,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        head_dim=64,
        ffn_dim=2304,
        window_size=512,
        dropout=0.05
    ):
        super().__init__()

        self.ln1=nn.LayerNorm(d_model)

        self.attn=M3GQAAttention(
            d_model=d_model,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            head_dim=head_dim,
            window_size=window_size,
            dropout=dropout
        )

        self.ln2=nn.LayerNorm(d_model)

        self.ffn=SwiGLU(
            d_model=d_model,
            ffn_dim=ffn_dim
        )

        self.dropout=nn.Dropout(dropout)

    def forward(self,x):

        x=x+self.dropout(
            self.attn(self.ln1(x))
        )

        x=x+self.dropout(
            self.ffn(self.ln2(x))
        )

        return x

class M3Model(nn.Module):
    def __init__(
        self,
        vocab_size=32000,
        d_model=768,
        n_layers=16,
        n_heads=12,
        n_kv_heads=4,
        head_dim=64,
        ffn_dim=2304,
        block_size=2048,
        window_size=512,
        dropout=0.05
    ):
        super().__init__()

        self.vocab_size=vocab_size
        self.d_model=d_model
        self.block_size=block_size
        self.tok_emb=nn.Embedding(vocab_size,d_model)
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
        self.blocks=nn.ModuleList([
            M3Block(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                head_dim=head_dim,
                ffn_dim=ffn_dim,
                window_size=window_size,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.ln_f=nn.LayerNorm(d_model)

        # Tied token embedding / LM head.
        self.lm_head=nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        self.lm_head.weight=self.tok_emb.weight

    def forward(self,idx):

        x=self.tok_emb(idx)

        for block in self.blocks:
            x=block(x)

        x=self.ln_f(x)

        return F.linear(
            x,
            self.tok_emb.weight
        )

In [7]:
model=M3Model()

params=sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total parameters: {params:,}")
print(f"Total parameters: {params/1e6:.4f} M")

print("\nEmbedding parameters:",model.tok_emb.weight.numel())
print("Block parameters:",sum(
    p.numel()
    for block in model.blocks
    for p in block.parameters()
))

Total parameters: 134,727,168
Total parameters: 134.7272 M

Embedding parameters: 24576000
Block parameters: 110149632


In [8]:
import torch,time

device=torch.device("cuda:0")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

model=M3Model().to(device).half()
model.train()

optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1
)

BATCH_SIZE=5
T=2048

x=torch.randint(
    0,
    VOCAB_SIZE,
    (BATCH_SIZE,T),
    device=device
)

y=torch.randint(
    0,
    VOCAB_SIZE,
    (BATCH_SIZE,T),
    device=device
)

torch.cuda.synchronize()
start=time.time()

logits=model(x)

loss=F.cross_entropy(
    logits.reshape(-1,VOCAB_SIZE),
    y.reshape(-1)
)

loss.backward()
optimizer.step()
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()
elapsed=time.time()-start

peak=torch.cuda.max_memory_allocated(device)/1e9
allocated=torch.cuda.memory_allocated(device)/1e9

print("Loss:",loss.item())
print("Time:",elapsed,"sec")
print("Tokens:",BATCH_SIZE*T)
print("Throughput:",(BATCH_SIZE*T)/elapsed,"tokens/sec")
print("Peak VRAM:",peak,"GB")
print("Current VRAM:",allocated,"GB")

Loss: 10.53125
Time: 1.4257168769836426 sec
Tokens: 10240
Throughput: 7182.351675365266 tokens/sec
Peak VRAM: 9.121204224 GB
Current VRAM: 2.091802112 GB


In [8]:
%%writefile /kaggle/working/m3_ddp_test.py
import os,time,torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.amp import autocast,GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP

VOCAB_SIZE=32000
BLOCK_SIZE=2048
D_MODEL=768
N_LAYERS=16
N_HEADS=12
N_KV_HEADS=4
HEAD_DIM=64
FFN_DIM=2304
WINDOW_SIZE=512
DROPOUT=0.05

os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

class RotaryEmbedding(nn.Module):
    def __init__(self,head_dim,max_seq_len=2048,base=10000):
        super().__init__()
        inv_freq=1.0/(base**(torch.arange(0,head_dim,2).float()/head_dim))
        pos=torch.arange(max_seq_len).float()
        freqs=torch.outer(pos,inv_freq)
        self.register_buffer("cos",freqs.cos(),persistent=False)
        self.register_buffer("sin",freqs.sin(),persistent=False)

    def forward(self,q,k):
        T=q.size(-2)
        cos=self.cos[:T].to(q.dtype)[None,None,:,:]
        sin=self.sin[:T].to(q.dtype)[None,None,:,:]

        q1=q[...,::2]
        q2=q[...,1::2]
        k1=k[...,::2]
        k2=k[...,1::2]

        q_rot=torch.stack(
            [q1*cos-q2*sin,q1*sin+q2*cos],
            dim=-1
        ).flatten(-2)

        k_rot=torch.stack(
            [k1*cos-k2*sin,k1*sin+k2*cos],
            dim=-1
        ).flatten(-2)

        return q_rot,k_rot

class M3GQAAttention(nn.Module):
    def __init__(
        self,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        head_dim=64,
        window_size=512,
        dropout=0.05
    ):
        super().__init__()

        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.head_dim=head_dim
        self.window_size=window_size
        self.dropout=dropout

        self.q_proj=nn.Linear(
            d_model,
            n_heads*head_dim,
            bias=False
        )

        self.k_proj=nn.Linear(
            d_model,
            n_kv_heads*head_dim,
            bias=False
        )

        self.v_proj=nn.Linear(
            d_model,
            n_kv_heads*head_dim,
            bias=False
        )

        self.o_proj=nn.Linear(
            n_heads*head_dim,
            d_model,
            bias=False
        )

        self.rope=RotaryEmbedding(
            head_dim,
            BLOCK_SIZE
        )

    def forward(self,x):

        B,T,C=x.shape

        q=self.q_proj(x)
        k=self.k_proj(x)
        v=self.v_proj(x)

        q=q.view(
            B,T,self.n_heads,self.head_dim
        ).transpose(1,2)

        k=k.view(
            B,T,self.n_kv_heads,self.head_dim
        ).transpose(1,2)

        v=v.view(
            B,T,self.n_kv_heads,self.head_dim
        ).transpose(1,2)

        q,k=self.rope(q,k)

        repeat_factor=self.n_heads//self.n_kv_heads

        k=k.repeat_interleave(
            repeat_factor,
            dim=1
        )

        v=v.repeat_interleave(
            repeat_factor,
            dim=1
        )

        positions=torch.arange(
            T,
            device=x.device
        )

        mask=(
            (positions[None,:]<=positions[:,None]) &
            (positions[None,:]>=(positions[:,None]-self.window_size+1))
        )

        mask=mask.unsqueeze(0).unsqueeze(0)

        y=F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0
        )

        y=y.transpose(1,2).contiguous().view(
            B,T,self.n_heads*self.head_dim
        )

        return self.o_proj(y)

class SwiGLU(nn.Module):
    def __init__(self,d_model=768,ffn_dim=2304):
        super().__init__()

        self.gate_proj=nn.Linear(
            d_model,
            ffn_dim,
            bias=False
        )

        self.up_proj=nn.Linear(
            d_model,
            ffn_dim,
            bias=False
        )

        self.down_proj=nn.Linear(
            ffn_dim,
            d_model,
            bias=False
        )

    def forward(self,x):
        return self.down_proj(
            F.silu(self.gate_proj(x))*self.up_proj(x)
        )

class M3Block(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1=nn.LayerNorm(D_MODEL)

        self.attn=M3GQAAttention(
            D_MODEL,
            N_HEADS,
            N_KV_HEADS,
            HEAD_DIM,
            WINDOW_SIZE,
            DROPOUT
        )

        self.ln2=nn.LayerNorm(D_MODEL)

        self.ffn=SwiGLU(
            D_MODEL,
            FFN_DIM
        )

        self.dropout=nn.Dropout(DROPOUT)

    def forward(self,x):

        x=x+self.dropout(
            self.attn(self.ln1(x))
        )

        x=x+self.dropout(
            self.ffn(self.ln2(x))
        )

        return x

class M3Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.tok_emb=nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
        self.blocks=nn.ModuleList([
            M3Block()
            for _ in range(N_LAYERS)
        ])

        self.ln_f=nn.LayerNorm(D_MODEL)

        self.lm_head=nn.Linear(
            D_MODEL,
            VOCAB_SIZE,
            bias=False
        )
        self.lm_head.weight=self.tok_emb.weight

    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)

        x=self.ln_f(x)
        return F.linear(x,self.tok_emb.weight)

def main():

    dist.init_process_group("nccl")
    rank=int(os.environ["RANK"])
    local_rank=int(os.environ["LOCAL_RANK"])
    world_size=int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    device=torch.device(f"cuda:{local_rank}")
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32=True
    model=M3Model().to(device)
    scaler=GradScaler("cuda")
    model=DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank
    )

    optimizer=torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=0.1,
        fused=True
    )

    BATCH_SIZE=2
    T=2048

    x=torch.randint(
        0,
        VOCAB_SIZE,
        (BATCH_SIZE,T),
        device=device
    )

    y=torch.randint(
        0,
        VOCAB_SIZE,
        (BATCH_SIZE,T),
        device=device
    )

    for _ in range(3):
        optimizer.zero_grad(set_to_none=True)
    
        with autocast("cuda",dtype=torch.float16):
            logits=model(x)
            loss=F.cross_entropy(
                logits.float().reshape(-1,VOCAB_SIZE),
                y.reshape(-1)
            )
    
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    torch.cuda.synchronize()
    dist.barrier()

    start=time.time()

    for _ in range(10):
        optimizer.zero_grad(set_to_none=True)
    
        with autocast("cuda", dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(
                logits.float().reshape(-1, VOCAB_SIZE),
                y.reshape(-1)
            )
    
        if not torch.isfinite(loss):
            print("Non-finite loss detected")
            break
    
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    torch.cuda.synchronize()
    dist.barrier()

    elapsed=time.time()-start

    tokens=10*BATCH_SIZE*world_size*T
    tok_per_sec=tokens/elapsed

    peak=torch.cuda.max_memory_allocated(
        device
    )/1e9

    if rank==0:

        print("="*60)
        print("M3 2-GPU DDP BENCHMARK")
        print("="*60)
        print("GPUs:",world_size)
        print("Batch/GPU:",BATCH_SIZE)
        print("Global batch:",BATCH_SIZE*world_size)
        print("Tokens/step:",BATCH_SIZE*world_size*T)
        print("10 steps:",elapsed,"sec")
        print("Throughput:",tok_per_sec,"tokens/sec")
        print("Peak VRAM GPU 0:",peak,"GB")
        print("Final loss:",loss.item())
        print("="*60)

    dist.destroy_process_group()

if __name__=="__main__":
    main()

Overwriting /kaggle/working/m3_ddp_test.py


In [9]:
!torchrun --nproc_per_node=2 /kaggle/working/m3_ddp_test.py

W1016 07:56:15.049000 202 torch/distributed/run.py:851] 
W1016 07:56:15.049000 202 torch/distributed/run.py:851] *****************************************
W1016 07:56:15.049000 202 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 07:56:15.049000 202 torch/distributed/run.py:851] *****************************************
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
M3 2-GPU DDP BENCHMARK
GPUs: 2
Batch/GPU: 2
Global batch: 4
Tokens/step: 8192
10 steps: 5.069038152694702 sec
Throughput: 16160.856859294165 tokens/sec
Peak VRAM GPU 0: 7.06434304 GB
Final loss: 8.014719009399414


In [18]:

%%writefile /kaggle/working/train_m3.py
import os, math, time
import torch
import torch.nn.functional as F
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast, GradScaler
from tqdm import tqdm

from m3_ddp_test import M3Model, VOCAB_SIZE, BLOCK_SIZE

DATA_DIR = "/kaggle/input/datasets/spikie/capstone-split"
CKPT_DIR = "/kaggle/working/m3_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_BIN = f"{DATA_DIR}/train_tokens.bin"
VALID_BIN = f"{DATA_DIR}/valid_tokens.bin"

BATCH_SIZE = 2
NUM_WORKERS = 2
TARGET_TOKENS = 250_000_000
LR = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
VAL_EVERY = 1000
VAL_BATCHES = 50
SAVE_EVERY = 500
RESUME = True
RESUME_PATH = f"{CKPT_DIR}/latest.pt"

SMOKE_TEST = False
SMOKE_STEPS = 5


class TokenBlockDataset(Dataset):
    def __init__(self, path):
        self.data = torch.from_file(
            path,
            dtype=torch.int16,
            size=os.path.getsize(path) // 2
        )
        self.n_blocks = (len(self.data) - 1) // BLOCK_SIZE

    def __len__(self):
        return self.n_blocks

    def __getitem__(self, idx):
        start = idx * BLOCK_SIZE
        block = self.data[start:start + BLOCK_SIZE + 1].to(torch.long)
        return block[:-1], block[1:]


def validate(model, loader, device):
    model.eval()
    total_loss = torch.zeros((), device=device)
    total_tokens = torch.zeros((), device=device)

    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= VAL_BATCHES:
                break

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = F.cross_entropy(
                    logits.float().reshape(-1, VOCAB_SIZE),
                    y.reshape(-1)
                )

            n = y.numel()
            total_loss += loss * n
            total_tokens += n

    dist.all_reduce(total_loss)
    dist.all_reduce(total_tokens)
    model.train()
    return (total_loss / total_tokens.clamp_min(1)).item()


def main():
    dist.init_process_group("nccl")
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.manual_seed(42)

    train_ds = TokenBlockDataset(TRAIN_BIN)
    valid_ds = TokenBlockDataset(VALID_BIN)

    train_sampler = DistributedSampler(
        train_ds, num_replicas=world_size, rank=rank, shuffle=True
    )
    valid_sampler = DistributedSampler(
        valid_ds, num_replicas=world_size, rank=rank, shuffle=False
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
    )
    valid_loader = DataLoader(
        valid_ds, batch_size=BATCH_SIZE, sampler=valid_sampler,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
    )

    model = M3Model().to(device)
    model = DDP(model, device_ids=[local_rank])

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
        fused=True
    )
    scaler = GradScaler("cuda")

    tokens_per_step = BATCH_SIZE * world_size * BLOCK_SIZE
    max_steps = math.ceil(TARGET_TOKENS / tokens_per_step)

    if SMOKE_TEST:
        max_steps = SMOKE_STEPS

    warmup_steps = max(1, int(0.03 * max_steps))
    step = 0
    best_val = float("inf")

    if RESUME and os.path.exists(RESUME_PATH):
        checkpoint = torch.load(
            RESUME_PATH, map_location="cpu", weights_only=False
        )
        model.module.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        step = checkpoint["step"]
        best_val = checkpoint["best_val"]

        if rank == 0:
            print(f"Resumed from step {step}")
            print(f"Previous best validation loss: {best_val:.4f}")
    elif rank == 0:
        print("Starting training from scratch.")

    if step >= max_steps:
        if rank == 0:
            print("Target step count already reached.")
        dist.destroy_process_group()
        return

    if rank == 0:
        print("Train blocks:", len(train_ds))
        print("Validation blocks:", len(valid_ds))
        print("GPUs:", world_size)
        print("Tokens per optimizer step:", tokens_per_step)
        print("Target tokens:", f"{TARGET_TOKENS:,}")
        print("Steps planned:", max_steps)
        print("Remaining steps:", max_steps - step)

    model.train()
    progress_bar = tqdm(
        total=max_steps,
        initial=step,
        desc="M3 Training",
        unit="step",
        dynamic_ncols=True,
        disable=(rank != 0)
    )

    run_start = time.time()
    last_log_time = run_start
    last_log_step = step
    running_loss = 0.0
    running_count = 0

    while step < max_steps:
        train_sampler.set_epoch(step // max(1, len(train_loader)))

        for x, y in train_loader:
            if step >= max_steps:
                break

            if step < warmup_steps:
                lr = LR * (step + 1) / warmup_steps
            else:
                progress = min(
                    1.0,
                    (step - warmup_steps) /
                    max(1, max_steps - warmup_steps)
                )
                lr = MIN_LR + 0.5 * (LR - MIN_LR) * (
                    1 + math.cos(math.pi * progress)
                )

            for group in optimizer.param_groups:
                group["lr"] = lr

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = F.cross_entropy(
                    logits.float().reshape(-1, VOCAB_SIZE),
                    y.reshape(-1)
                )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at step {step + 1}")

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            step += 1
            running_loss += loss.item()
            running_count += 1

            # Validation on all ranks; only rank 0 displays/saves.
            val_loss = None
            if step % VAL_EVERY == 0 or step == max_steps:
                val_loss = validate(model.module, valid_loader, device)

                if rank == 0:
                    print(f"\nStep {step} | validation loss: {val_loss:.4f}")

                    if val_loss < best_val:
                        best_val = val_loss
                        torch.save(
                            model.module.state_dict(),
                            f"{CKPT_DIR}/best.pt"
                        )

            # Save full state for resuming.
            if step % SAVE_EVERY == 0 or step == max_steps:
                if rank == 0:
                    save_path = f"{CKPT_DIR}/latest_tmp.pt"
                    torch.save(
                        {
                            "step": step,
                            "model": model.module.state_dict(),
                            "optimizer": optimizer.state_dict(),
                            "scaler": scaler.state_dict(),
                            "best_val": best_val
                        },
                        save_path
                    )
                    os.replace(save_path, RESUME_PATH)

            if rank == 0:
                progress_bar.update(1)

                now = time.time()
                if step % 10 == 0 or step == max_steps:
                    elapsed = now - run_start
                    interval = now - last_log_time
                    steps_interval = step - last_log_step
                    steps_per_sec = steps_interval / max(interval, 1e-9)
                    remaining = max_steps - step
                    eta_minutes = remaining / max(steps_per_sec, 1e-9) / 60

                    avg_loss = running_loss / max(1, running_count)
                    progress_bar.set_postfix({
                        "loss": f"{avg_loss:.4f}",
                        "lr": f"{lr:.2e}",
                        "tok/s": f"{steps_per_sec * tokens_per_step:.0f}",
                        "ETA": f"{eta_minutes:.1f}m"
                    })
                    running_loss = 0.0
                    running_count = 0
                    last_log_time = now
                    last_log_step = step

            if step >= max_steps:
                break

    dist.barrier()
    if rank == 0:
        progress_bar.close()
        print("Training run complete.")
        print("Elapsed minutes:", round((time.time() - run_start) / 60, 2))
        print("Best validation loss:", best_val)
        print("Checkpoints:", CKPT_DIR)

    dist.destroy_process_group()


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/train_m3.py


In [19]:
!torchrun --nproc_per_node=2 /kaggle/working/train_m3.py

W1016 08:12:59.208000 533 torch/distributed/run.py:851] 
W1016 08:12:59.208000 533 torch/distributed/run.py:851] *****************************************
W1016 08:12:59.208000 533 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1016 08:12:59.208000 533 torch/distributed/run.py:851] *****************************************
Resumed from step 5
Previous best validation loss: 6.1824
Train blocks: 2799570
Validation blocks: 312738
GPUs: 2
Tokens per optimizer step: 8192
Target tokens: 250,000,000
Steps planned: 30518
Remaining steps: 30513
M3 Training:   3%| | 999/30518 [09:42<4:48:54,  1.70step/s, loss=3.2251, lr=3.00
Step 1000 | validation loss: 3.1481
M3 Training:   7%| | 1999/30518 [19:42<4:38:57,  1.70step/s, loss=2.5678, lr=2.9
Step 2000 | validation loss: 2.7140
M3 Training:  10%| | 2999/3051

Phase 3

In [26]:
import os
import json
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

REPO_ID = "Exploration-Lab/CS-781-Capstone"
REPO_DIR = "Phase_3_RR"
DATA_DIR = Path("/kaggle/working/phase3_data/Phase_3_RR")

files_to_download = [
    "Train/CL_train.json",
    "Train/CL_dev.json",
    "Train/IT_train.json",
    "Train/IT_dev.json",
    "Train/label_vocab.json",
    "Test/CL_test.json",
    "Test/IT_test.json",
]

DATA_DIR.mkdir(parents=True, exist_ok=True)

for filename in files_to_download:
    print(f"Downloading {filename}...")

    downloaded_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=f"{REPO_DIR}/{filename}",
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir="/kaggle/working/hf_phase3_download"
    )

    destination = DATA_DIR / filename
    destination.parent.mkdir(parents=True, exist_ok=True)

    with open(downloaded_path, "rb") as src:
        with open(destination, "wb") as dst:
            dst.write(src.read())

    print(f"Saved: {destination}")

# Verify the downloaded JSON files
print("\n=== Verification ===")

for filename in files_to_download:
    path = DATA_DIR / filename
    assert path.is_file(), f"Missing file: {path}"

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if filename.endswith("_train.json") or filename.endswith("_dev.json"):
        print(f"{filename}: {len(data)} labeled records")
    elif filename.endswith("_test.json"):
        print(f"{filename}: {len(data)} test documents")
    else:
        print(f"{filename}: loaded successfully")

print("\nAll Phase 3 files downloaded and verified.")

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Saved: /kaggle/working/phase3_data/Phase_3_RR/Train/CL_train.json


CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

Saved: /kaggle/working/phase3_data/Phase_3_RR/Train/CL_dev.json


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

Saved: /kaggle/working/phase3_data/Phase_3_RR/Train/IT_train.json


... (4 outputs left out)

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

Saved: /kaggle/working/phase3_data/Phase_3_RR/Test/CL_test.json


IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

Saved: /kaggle/working/phase3_data/Phase_3_RR/Test/IT_test.json

=== Verification ===
Train/CL_train.json: 40 labeled records
Train/CL_dev.json: 5 labeled records
Train/IT_train.json: 40 labeled records
Train/IT_dev.json: 5 labeled records
Train/label_vocab.json: loaded successfully
Test/CL_test.json: 5 test documents
Test/IT_test.json: 5 test documents

All Phase 3 files downloaded and verified.


In [30]:
import os
from pathlib import Path
import torch
import pickle

# Existing Phase 1 artifacts
checkpoint_dir = Path("/kaggle/working/m3_checkpoints")
best_path = checkpoint_dir / "best.pt"
latest_path = checkpoint_dir / "latest.pt"

# Existing tokenizer artifacts
tokenizer_dir = Path("/kaggle/input/datasets/spikie/capstone-split")
tokenizer_paths = [
    tokenizer_dir / "legal_bpe_32k.pkl",
    tokenizer_dir / "legal_bpe_64k.pkl",
]

print("=== CHECKPOINT FILES ===")
for path in [best_path, latest_path]:
    print(f"\n{path}")
    print("Exists:", path.exists())
    if path.exists():
        print(f"Size: {path.stat().st_size / (1024**3):.3f} GiB")

print("\n=== TOKENIZER FILES ===")
for path in tokenizer_paths:
    print(f"\n{path}")
    print("Exists:", path.exists())
    if path.exists():
        print(f"Size: {path.stat().st_size / (1024**2):.2f} MiB")

print("\n=== BEST CHECKPOINT STRUCTURE ===")
if best_path.exists():
    state = torch.load(
        best_path,
        map_location="cpu",
        weights_only=True
    )

    print("Object type:", type(state).__name__)
    print("Number of entries:", len(state))

    for i, (key, value) in enumerate(state.items()):
        if i == 12:
            break
        if torch.is_tensor(value):
            print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")
        else:
            print(f"{key}: {type(value).__name__}")

    del state

print("\n=== LATEST CHECKPOINT METADATA ===")
if latest_path.exists():
    latest = torch.load(
        latest_path,
        map_location="cpu",
        weights_only=False
    )
    print("Top-level keys:", list(latest.keys()))
    for key in ["step", "best_val"]:
        if key in latest:
            print(f"{key}: {latest[key]}")
    if "model" in latest:
        print("Model state entries:", len(latest["model"]))
    del latest

print("\nInspection complete.")

=== CHECKPOINT FILES ===

/kaggle/working/m3_checkpoints/best.pt
Exists: True
Size: 0.502 GiB

/kaggle/working/m3_checkpoints/latest.pt
Exists: True
Size: 1.506 GiB

=== TOKENIZER FILES ===

/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl
Exists: True
Size: 2.16 MiB

/kaggle/input/datasets/spikie/capstone-split/legal_bpe_64k.pkl
Exists: True
Size: 4.40 MiB

=== BEST CHECKPOINT STRUCTURE ===
Object type: OrderedDict
Number of entries: 180
tok_emb.weight: shape=(32000, 768), dtype=torch.float32
blocks.0.ln1.weight: shape=(768,), dtype=torch.float32
blocks.0.ln1.bias: shape=(768,), dtype=torch.float32
blocks.0.attn.q_proj.weight: shape=(768, 768), dtype=torch.float32
blocks.0.attn.k_proj.weight: shape=(256, 768), dtype=torch.float32
blocks.0.attn.v_proj.weight: shape=(256, 768), dtype=torch.float32
blocks.0.attn.o_proj.weight: shape=(768, 768), dtype=torch.float32
blocks.0.ln2.weight: shape=(768,), dtype=torch.float32
blocks.0.ln2.bias: shape=(768,), dtype=torch.float32
blo

In [35]:
import inspect

print("=== TOKENIZER METHOD SIGNATURES ===")
print("encode_text:", inspect.signature(tokenizer.encode_text))
print("encode_word:", inspect.signature(tokenizer.encode_word))

sample = (
    "The facts as stated in the complaint are important "
    "for deciding the present issue."
)

print("\n=== ENCODE SAMPLE TEXT ===")
encoded = tokenizer.encode_text(sample)

print("Output type:", type(encoded))
print("Output:", encoded)

if isinstance(encoded, (list, tuple)):
    print("Number of tokens:", len(encoded))
    print("First 20 token IDs:", encoded[:20])

print("\n=== SPECIAL TOKEN INFORMATION ===")
for name in dir(tokenizer):
    if any(term in name.lower() for term in ["pad", "unk", "bos", "eos", "special"]):
        try:
            value = getattr(tokenizer, name)
            if not callable(value):
                print(f"{name} = {value}")
        except Exception:
            pass

=== TOKENIZER METHOD SIGNATURES ===
encode_text: (text, add_eos=True)
encode_word: (word)

=== ENCODE SAMPLE TEXT ===
Output type: <class 'list'>
Output: [323, 33, 1059, 33, 272, 33, 885, 33, 260, 33, 258, 33, 1165, 33, 401, 33, 6048, 33, 290, 33, 5522, 33, 258, 33, 546, 33, 10581, 0]
Number of tokens: 28
First 20 token IDs: [323, 33, 1059, 33, 272, 33, 885, 33, 260, 33, 258, 33, 1165, 33, 401, 33, 6048, 33, 290, 33]

=== SPECIAL TOKEN INFORMATION ===
SPECIALS = ['<eos>']
eos_id = 0


In [38]:
import json
import numpy as np
from pathlib import Path

DATA_DIR = Path("/kaggle/working/phase3_data/Phase_3_RR")

def load_labeled_documents(filenames):
    documents = []

    for filename in filenames:
        path = DATA_DIR / filename

        with open(path, "r", encoding="utf-8") as f:
            records = json.load(f)

        for record in records:
            documents.append({
                "id": record["id"],
                "sentences": record["text"],
                "labels": record["labels"]
            })

    return documents

train_docs = load_labeled_documents([
    "Train/CL_train.json",
    "Train/IT_train.json"
])

dev_docs = load_labeled_documents([
    "Train/CL_dev.json",
    "Train/IT_dev.json"
])

with open(DATA_DIR / "Train/label_vocab.json", "r", encoding="utf-8") as f:
    label_vocab = json.load(f)

# Tokenize each sentence independently.
train_examples = []
dev_examples = []

for docs, output in [
    (train_docs, train_examples),
    (dev_docs, dev_examples)
]:
    for doc in docs:
        for sentence, label in zip(doc["sentences"], doc["labels"]):
            token_ids = tokenizer.encode_text(sentence, add_eos=True)

            output.append({
                "doc_id": doc["id"],
                "text": sentence,
                "input_ids": token_ids,
                "label": label_vocab[label]
            })

train_lengths = np.array([
    len(example["input_ids"]) for example in train_examples
])

dev_lengths = np.array([
    len(example["input_ids"]) for example in dev_examples
])

print("Training documents:", len(train_docs))
print("Training sentence examples:", len(train_examples))
print("Validation documents:", len(dev_docs))
print("Validation sentence examples:", len(dev_examples))
print("Number of classes:", len(label_vocab))

print("\n=== Token length statistics ===")
for name, lengths in [
    ("Train", train_lengths),
    ("Dev", dev_lengths)
]:
    print(f"\n{name}:")
    print("  Min:", int(lengths.min()))
    print("  Median:", float(np.median(lengths)))
    print("  90th percentile:", float(np.percentile(lengths, 90)))
    print("  95th percentile:", float(np.percentile(lengths, 95)))
    print("  99th percentile:", float(np.percentile(lengths, 99)))
    print("  Maximum:", int(lengths.max()))

print("\nTokenization complete.")

Training documents: 80
Training sentence examples: 16845
Validation documents: 10
Validation sentence examples: 2142
Number of classes: 13

=== Token length statistics ===

Train:
  Min: 2
  Median: 58.0
  90th percentile: 129.0
  95th percentile: 161.0
  99th percentile: 293.5600000000013
  Maximum: 2190

Dev:
  Min: 2
  Median: 59.0
  90th percentile: 123.0
  95th percentile: 156.94999999999982
  99th percentile: 270.59000000000015
  Maximum: 744

Tokenization complete.


In [40]:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 256
BATCH_SIZE = 8
PAD_ID = tokenizer.eos_id  # Padding positions will be identified by the mask.

class RhetoricalDataset(Dataset):
    def __init__(self, examples, max_len=256):
        self.examples = examples
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        example = self.examples[idx]
        input_ids = example["input_ids"][:self.max_len]

        return {
            "input_ids": input_ids,
            "label": example["label"],
            "length": len(input_ids),
            "doc_id": example["doc_id"]
        }

def collate_rhetorical(batch):
    max_len = max(item["length"] for item in batch)

    input_ids = torch.full(
        (len(batch), max_len),
        PAD_ID,
        dtype=torch.long
    )
    attention_mask = torch.zeros(
        (len(batch), max_len),
        dtype=torch.bool
    )
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    for i, item in enumerate(batch):
        length = item["length"]
        input_ids[i, :length] = torch.tensor(
            item["input_ids"],
            dtype=torch.long
        )
        attention_mask[i, :length] = True

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "doc_ids": [item["doc_id"] for item in batch]
    }

train_dataset = RhetoricalDataset(train_examples, MAX_LEN)
dev_dataset = RhetoricalDataset(dev_examples, MAX_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_rhetorical
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rhetorical
)

batch = next(iter(train_loader))

print("Train examples:", len(train_dataset))
print("Dev examples:", len(dev_dataset))
print("Batch input shape:", tuple(batch["input_ids"].shape))
print("Batch mask shape:", tuple(batch["attention_mask"].shape))
print("Batch labels shape:", tuple(batch["labels"].shape))
print("First batch labels:", batch["labels"].tolist())
print("Padding positions:", (~batch["attention_mask"]).sum().item())
print("All labels in range:",
      bool(((batch["labels"] >= 0) & (batch["labels"] < 13)).all()))

Train examples: 16845
Dev examples: 2142
Batch input shape: (8, 115)
Batch mask shape: (8, 115)
Batch labels shape: (8,)
First batch labels: [2, 4, 10, 2, 2, 10, 0, 2]
Padding positions: 401
All labels in range: True


In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRMAPlusLinear(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()

        self.base = base_layer
        self.rank = rank
        self.scale = alpha / rank

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        # Freeze the original pretrained linear layer.
        for param in self.base.parameters():
            param.requires_grad = False

        # A maps the base output into a low-rank representation.
        self.A = nn.Parameter(
            torch.empty(rank, out_features)
        )

        # B maps the low-rank representation back to output space.
        self.B = nn.Parameter(
            torch.zeros(out_features, rank)
        )

        nn.init.normal_(self.A, mean=0.0, std=0.02)

    def forward(self, x):
        base_output = self.base(x)
        adaptation = F.linear(
            F.linear(base_output, self.A),
            self.B
        )
        return base_output + self.scale * adaptation


# Test the wrapper independently before modifying the pretrained model.
torch.manual_seed(42)

test_base = nn.Linear(768, 768)
test_input = torch.randn(2, 10, 768)

with torch.no_grad():
    original_output = test_base(test_input)

test_layer = LoRMAPlusLinear(
    test_base,
    rank=8,
    alpha=16
)

with torch.no_grad():
    adapted_output = test_layer(test_input)

max_difference = (
    original_output - adapted_output
).abs().max().item()

trainable_parameters = sum(
    p.numel() for p in test_layer.parameters()
    if p.requires_grad
)

print("Original output shape:", tuple(original_output.shape))
print("Adapted output shape:", tuple(adapted_output.shape))
print("Maximum initial output difference:", max_difference)
print("LoRMA+ trainable parameters:", trainable_parameters)
print("Expected trainable parameters:", 2 * 768 * 8)
print("Original layer frozen:", all(
    not p.requires_grad for p in test_layer.base.parameters()
))

Original output shape: (2, 10, 768)
Adapted output shape: (2, 10, 768)
Maximum initial output difference: 0.0
LoRMA+ trainable parameters: 12288
Expected trainable parameters: 12288
Original layer frozen: True


In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# Start with the pretrained checkpoint already loaded into base_model.
# Do not reload or overwrite the Phase 1 checkpoint.

NUM_LABELS = len(label_vocab)
LORMA_RANK = 8
LORMA_ALPHA = 16
CLASSIFIER_DROPOUT = 0.1

class M3Classifier(nn.Module):
    def __init__(self, pretrained_model, num_labels, rank=8, alpha=16):
        super().__init__()

        self.tok_emb = pretrained_model.tok_emb
        self.blocks = pretrained_model.blocks
        self.ln_f = pretrained_model.ln_f

        # Freeze all pretrained backbone parameters first.
        for param in self.tok_emb.parameters():
            param.requires_grad = False

        for param in self.blocks.parameters():
            param.requires_grad = False

        for param in self.ln_f.parameters():
            param.requires_grad = False

        # Replace q_proj and o_proj in every transformer block.
        self.adapted_layers = []

        for block_idx in range(len(self.blocks)):
            block = self.blocks[block_idx]

            block.attn.q_proj = LoRMAPlusLinear(
                block.attn.q_proj,
                rank=rank,
                alpha=alpha
            )

            block.attn.o_proj = LoRMAPlusLinear(
                block.attn.o_proj,
                rank=rank,
                alpha=alpha
            )

            self.adapted_layers.append(
                f"blocks.{block_idx}.attn.q_proj"
            )
            self.adapted_layers.append(
                f"blocks.{block_idx}.attn.o_proj"
            )

        # Sentence-level classification head.
        self.dropout = nn.Dropout(CLASSIFIER_DROPOUT)
        self.classifier = nn.Linear(
            pretrained_model.tok_emb.embedding_dim,
            num_labels
        )

    def forward(self, input_ids, attention_mask):
        # input_ids: [batch, sequence]
        hidden = self.tok_emb(input_ids)

        for block in self.blocks:
            hidden = block(hidden)

        hidden = self.ln_f(hidden)

        # Mask-aware mean pooling: exclude padding positions.
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        hidden = hidden * mask

        pooled = hidden.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        logits = self.classifier(self.dropout(pooled))
        return logits


# Build classifier using the already-loaded pretrained model.
model = M3Classifier(
    pretrained_model=base_model,
    num_labels=NUM_LABELS,
    rank=LORMA_RANK,
    alpha=LORMA_ALPHA
).to(DEVICE)

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
lorma_parameters = sum(
    p.numel()
    for name, p in model.named_parameters()
    if p.requires_grad and (name.endswith(".A") or name.endswith(".B"))
)
classifier_parameters = sum(
    p.numel() for p in model.classifier.parameters()
)

print("Device:", DEVICE)
print("Number of labels:", NUM_LABELS)
print("Adapted projection layers:", len(model.adapted_layers))
print("First adapted layer:", model.adapted_layers[0])
print("Last adapted layer:", model.adapted_layers[-1])
print("Total model parameters:", f"{total_parameters:,}")
print("Total trainable parameters:", f"{trainable_parameters:,}")
print("LoRMA+ trainable parameters:", f"{lorma_parameters:,}")
print("Classifier trainable parameters:", f"{classifier_parameters:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.4f}%"
)

# Check that the frozen LM head is not part of the classifier.
print("Uses pretrained token embeddings:", model.tok_emb is base_model.tok_emb)
print("Uses pretrained transformer blocks:", model.blocks is base_model.blocks)
print("Uses pretrained final norm:", model.ln_f is base_model.ln_f)

Device: cuda
Number of labels: 13
Adapted projection layers: 32
First adapted layer: blocks.0.attn.q_proj
Last adapted layer: blocks.15.attn.o_proj
Total model parameters: 135,130,381
Total trainable parameters: 403,213
LoRMA+ trainable parameters: 393,216
Classifier trainable parameters: 9,997
Trainable percentage: 0.2984%
Uses pretrained token embeddings: True
Uses pretrained transformer blocks: True
Uses pretrained final norm: True


In [48]:
# Get one real batch from the existing training loader.
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

model.eval()
model.zero_grad(set_to_none=True)

# Test forward pass.
with torch.no_grad():
    logits = model(input_ids, attention_mask)

print("Input IDs shape:", tuple(input_ids.shape))
print("Attention mask shape:", tuple(attention_mask.shape))
print("Labels shape:", tuple(labels.shape))
print("Logits shape:", tuple(logits.shape))
print("Logits finite:", torch.isfinite(logits).all().item())
print("Labels in valid range:", (
    (labels >= 0) & (labels < NUM_LABELS)
).all().item())

# Test whether gradients reach LoRMA parameters and the classifier.
model.train()
model.zero_grad(set_to_none=True)

logits = model(input_ids, attention_mask)
loss = F.cross_entropy(logits, labels)
loss.backward()

lorma_grad_count = 0
lorma_nonzero_grad_count = 0
classifier_grad_count = 0

for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        if name.endswith(".A") or name.endswith(".B"):
            lorma_grad_count += 1
            if param.grad.abs().sum().item() > 0:
                lorma_nonzero_grad_count += 1

        if name.startswith("classifier."):
            classifier_grad_count += 1

print("\nLoss:", loss.item())
print("LoRMA parameters with gradients:", lorma_grad_count)
print("LoRMA parameters with nonzero gradients:", lorma_nonzero_grad_count)
print("Classifier parameters with gradients:", classifier_grad_count)

# Clear test gradients before training setup.
model.zero_grad(set_to_none=True)

Input IDs shape: (8, 213)
Attention mask shape: (8, 213)
Labels shape: (8,)
Logits shape: (8, 13)
Logits finite: True
Labels in valid range: True

Loss: 2.559706687927246
LoRMA parameters with gradients: 64
LoRMA parameters with nonzero gradients: 32
Classifier parameters with gradients: 2


In [49]:
import numpy as np
from collections import Counter
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# Save M3 outputs separately from the Phase 1 checkpoint.
M3_DIR = Path("/kaggle/working/phase3_m3_lorma")
M3_DIR.mkdir(parents=True, exist_ok=True)

NUM_EPOCHS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Count labels from training examples only.
train_label_counts = Counter(
    example["label"] for example in train_examples
)

print("Training label distribution:")
for label, label_id in sorted(label_vocab.items(), key=lambda x: x[1]):
    print(f"{label_id:2d} | {label:28s} | {train_label_counts[label_id]}")

# Inverse-frequency weights, normalized to mean 1 and capped.
counts = torch.tensor(
    [train_label_counts[i] for i in range(NUM_LABELS)],
    dtype=torch.float32
)

class_weights = counts.sum() / (
    NUM_LABELS * counts.clamp(min=1.0)
)

# Cap rare-class weights to limit unstable gradients.
class_weights = class_weights.clamp(max=5.0)
class_weights = class_weights / class_weights.mean()

class_weights = class_weights.to(DEVICE)

print("\nClass weights:")
for label, label_id in sorted(label_vocab.items(), key=lambda x: x[1]):
    print(f"{label_id:2d} | {label:28s} | {class_weights[label_id].item():.3f}")

# Only trainable parameters enter the optimizer.
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print("\nTraining configuration:")
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Max gradient norm:", MAX_GRAD_NORM)
print("Optimizer parameters:", sum(
    p.numel() for group in optimizer.param_groups
    for p in group["params"]
))
print("Output directory:", M3_DIR)

Training label distribution:
 0 | Fact                         | 4098
 1 | Issue                        | 107
 2 | ArgumentPetitioner           | 1613
 3 | ArgumentRespondent           | 1348
 4 | Statute                      | 1099
 5 | Dissent                      | 284
 6 | PrecedentReliedUpon          | 1945
 7 | PrecedentNotReliedUpon       | 569
 8 | PrecedentOverruled           | 3
 9 | RulingByLowerCourt           | 874
10 | RatioOfTheDecision           | 4301
11 | RulingByPresentCourt         | 307
12 | None                         | 297

Class weights:
 0 | Fact                         | 0.132
 1 | Issue                        | 2.088
 2 | ArgumentPetitioner           | 0.335
 3 | ArgumentRespondent           | 0.401
 4 | Statute                      | 0.492
 5 | Dissent                      | 1.905
 6 | PrecedentReliedUpon          | 0.278
 7 | PrecedentNotReliedUpon       | 0.951
 8 | PrecedentOverruled           | 2.088
 9 | RulingByLowerCourt           | 0.619
10 | RatioO

In [53]:
from tqdm.auto import tqdm
import time
import torch
from sklearn.metrics import accuracy_score, f1_score

def run_train_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total_examples = 0
    all_predictions = []
    all_labels = []

    start_time = time.time()

    progress = tqdm(loader, desc="Training", leave=True)

    for batch in progress:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            MAX_GRAD_NORM
        )

        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        predictions = logits.argmax(dim=1)
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_labels.extend(labels.detach().cpu().tolist())

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss / total_examples:.4f}"
        )

    return {
        "loss": total_loss / max(total_examples, 1),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels, all_predictions,
            average="macro", zero_division=0
        ),
        "seconds": time.time() - start_time
    }


@torch.no_grad()
def run_evaluation(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_examples = 0
    all_predictions = []
    all_labels = []

    progress = tqdm(loader, desc="Validation", leave=True)

    for batch in progress:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        all_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss / total_examples:.4f}"
        )

    return {
        "loss": total_loss / max(total_examples, 1),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels, all_predictions,
            average="macro", zero_division=0
        ),
        "predictions": all_predictions,
        "labels": all_labels
    }

In [54]:
import json
import time
import torch
from pathlib import Path

history = []
best_val_macro_f1 = -1.0

epoch = 1
print(f"Starting LoRMA+ epoch {epoch}/{NUM_EPOCHS}")
train_metrics = run_train_epoch(model, train_loader, optimizer, criterion)
val_metrics = run_evaluation(model, dev_loader, criterion)
record = {
    "epoch": epoch,
    "train_loss": train_metrics["loss"],
    "train_accuracy": train_metrics["accuracy"],
    "train_macro_f1": train_metrics["macro_f1"],
    "val_loss": val_metrics["loss"],
    "val_accuracy": val_metrics["accuracy"],
    "val_macro_f1": val_metrics["macro_f1"],
    "seconds": train_metrics["seconds"]
}
history.append(record)

print("\nEpoch results")
print(f"Train loss:      {record['train_loss']:.4f}")
print(f"Train accuracy:  {record['train_accuracy']:.4f}")
print(f"Train macro-F1:  {record['train_macro_f1']:.4f}")
print(f"Validation loss: {record['val_loss']:.4f}")
print(f"Validation acc:  {record['val_accuracy']:.4f}")
print(f"Validation F1:   {record['val_macro_f1']:.4f}")
print(f"Training time:   {record['seconds'] / 60:.2f} minutes")

# Save resumable checkpoint and training history.
checkpoint = {
    "epoch": epoch,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "history": history,
    "best_val_macro_f1": max(best_val_macro_f1, val_metrics["macro_f1"])
}

torch.save(checkpoint, M3_DIR / "latest.pt")

if val_metrics["macro_f1"] > best_val_macro_f1:
    best_val_macro_f1 = val_metrics["macro_f1"]
    torch.save(checkpoint, M3_DIR / "best.pt")

with open(M3_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nSaved checkpoints in:", M3_DIR)
print("CUDA peak allocated memory (GB):",
      round(torch.cuda.max_memory_allocated() / (1024**3), 2))

Starting LoRMA+ epoch 1/8


Training:   0%|          | 0/2106 [00:00<?, ?it/s]

Validation:   0%|          | 0/268 [00:00<?, ?it/s]


Epoch results
Train loss:      1.6521
Train accuracy:  0.4112
Train macro-F1:  0.3515
Validation loss: 1.8361
Validation acc:  0.3679
Validation F1:   0.3238
Training time:   7.84 minutes

Saved checkpoints in: /kaggle/working/phase3_m3_lorma
CUDA peak allocated memory (GB): 2.25


In [55]:
import json
import time
import torch
from tqdm.auto import tqdm

# Continue from the in-memory state after epoch 1.
start_epoch = history[-1]["epoch"] + 1

best_val_macro_f1 = max(
    record["val_macro_f1"] for record in history
)

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f"\n{'=' * 20} Epoch {epoch}/{NUM_EPOCHS} {'=' * 20}")

    train_metrics = run_train_epoch(
        model, train_loader, optimizer, criterion
    )

    val_metrics = run_evaluation(
        model, dev_loader, criterion
    )

    record = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "seconds": train_metrics["seconds"]
    }
    history.append(record)

    print(f"\nTrain loss:      {record['train_loss']:.4f}")
    print(f"Train accuracy:  {record['train_accuracy']:.4f}")
    print(f"Train macro-F1:  {record['train_macro_f1']:.4f}")
    print(f"Validation loss: {record['val_loss']:.4f}")
    print(f"Validation acc:  {record['val_accuracy']:.4f}")
    print(f"Validation F1:   {record['val_macro_f1']:.4f}")
    print(f"Training time:   {record['seconds'] / 60:.2f} minutes")

    improved = record["val_macro_f1"] > best_val_macro_f1
    if improved:
        best_val_macro_f1 = record["val_macro_f1"]

    checkpoint = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "history": history,
        "best_val_macro_f1": best_val_macro_f1
    }

    torch.save(checkpoint, M3_DIR / "latest.pt")

    if improved:
        torch.save(checkpoint, M3_DIR / "best.pt")
        print("Saved new best checkpoint.")

    with open(M3_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    print("Best validation macro-F1 so far:", f"{best_val_macro_f1:.4f}")

print("\nTraining complete.")
print("Best checkpoint:", M3_DIR / "best.pt")
print("Latest checkpoint:", M3_DIR / "latest.pt")


==================== Epoch 2/8 ====================


Training:   0%|          | 0/2106 [00:00<?, ?it/s]

Validation:   0%|          | 0/268 [00:00<?, ?it/s]


Train loss:      1.4197
Train accuracy:  0.4662
Train macro-F1:  0.4214
Validation loss: 1.8657
Validation acc:  0.3861
Validation F1:   0.3255
Training time:   7.90 minutes
Saved new best checkpoint.
Best validation macro-F1 so far: 0.3255

==================== Epoch 3/8 ====================


Training:   0%|          | 0/2106 [00:00<?, ?it/s]

Validation:   0%|          | 0/268 [00:00<?, ?it/s]

... (12 outputs left out)


Train loss:      0.9240
Train accuracy:  0.6104
Train macro-F1:  0.6363
Validation loss: 2.3060
Validation acc:  0.3754
Validation F1:   0.3296
Training time:   7.88 minutes
Best validation macro-F1 so far: 0.3350

==================== Epoch 8/8 ====================


Training:   0%|          | 0/2106 [00:00<?, ?it/s]

Validation:   0%|          | 0/268 [00:00<?, ?it/s]


Train loss:      0.8381
Train accuracy:  0.6277
Train macro-F1:  0.6746
Validation loss: 2.3590
Validation acc:  0.3959
Validation F1:   0.3378
Training time:   7.86 minutes
Saved new best checkpoint.
Best validation macro-F1 so far: 0.3378

Training complete.
Best checkpoint: /kaggle/working/phase3_m3_lorma/best.pt
Latest checkpoint: /kaggle/working/phase3_m3_lorma/latest.pt


In [69]:
import json
from pathlib import Path
from tqdm.auto import tqdm
import torch

TEST_DIR = Path("/kaggle/working/phase3_data/Phase_3_RR/Test")
OUT_PATH = Path("/kaggle/working/phase3_m3_lorma/prediction.json")

MAX_LEN = 256
PAD_ID = tokenizer.eos_id

ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

def encode_test_sentence(sentence):
    ids = tokenizer.encode_text(sentence, add_eos=True)

    # Support tokenizers that return either a list or an object.
    if hasattr(ids, "ids"):
        ids = ids.ids

    ids = list(ids)[:MAX_LEN]

    if len(ids) == 0:
        ids = [PAD_ID]

    return ids

predictions = {}
model.eval()

for filename in ["CL_test.json", "IT_test.json"]:
    with open(TEST_DIR / filename, "r", encoding="utf-8") as f:
        test_docs = json.load(f)

    print(f"\nPredicting {filename}: {len(test_docs)} documents")

    for doc_id, sentences in tqdm(test_docs.items(), desc=filename):
        doc_predictions = []

        for sentence in sentences:
            input_ids_list = encode_test_sentence(sentence)

            input_ids = torch.tensor(
                [input_ids_list],
                dtype=torch.long,
                device=DEVICE
            )
            attention_mask = torch.ones_like(
                input_ids, dtype=torch.bool
            )

            with torch.inference_mode():
                logits = model(input_ids, attention_mask)
                predicted_id = int(logits.argmax(dim=-1).item())

            doc_predictions.append(ID_TO_LABEL[predicted_id])

        predictions[doc_id] = doc_predictions

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

# Validate document IDs and sentence counts against both test files.
assert len(predictions) == 10, f"Expected 10 documents, got {len(predictions)}"

for filename in ["CL_test.json", "IT_test.json"]:
    with open(TEST_DIR / filename, "r", encoding="utf-8") as f:
        source_docs = json.load(f)

    for doc_id, sentences in source_docs.items():
        assert doc_id in predictions, f"Missing document: {doc_id}"
        assert len(predictions[doc_id]) == len(sentences), (
            f"Sentence count mismatch for {doc_id}"
        )
        assert all(label in LABEL_TO_ID for label in predictions[doc_id])

print("\nPrediction file:", OUT_PATH)
print("Documents:", len(predictions))
print("Total sentences:", sum(len(v) for v in predictions.values()))
print("Label distribution:")
from collections import Counter
print(Counter(label for labels in predictions.values() for label in labels))
print("\nSample document:", next(iter(predictions)))
print("Sample predictions:", next(iter(predictions.values()))[:10])
print("Validation passed.")


Predicting CL_test.json: 5 documents


CL_test.json:   0%|          | 0/5 [00:00<?, ?it/s]


Predicting IT_test.json: 5 documents


IT_test.json:   0%|          | 0/5 [00:00<?, ?it/s]


Prediction file: /kaggle/working/phase3_m3_lorma/prediction.json
Documents: 10
Total sentences: 2197
Label distribution:
Counter({'RatioOfTheDecision': 504, 'Fact': 446, 'PrecedentReliedUpon': 277, 'Statute': 192, 'RulingByLowerCourt': 183, 'ArgumentPetitioner': 165, 'ArgumentRespondent': 128, 'PrecedentNotReliedUpon': 107, 'RulingByPresentCourt': 76, 'None': 64, 'Dissent': 41, 'Issue': 14})

Sample document: HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489
Sample predictions: ['None', 'Fact', 'Fact', 'Fact', 'ArgumentPetitioner', 'ArgumentRespondent', 'ArgumentPetitioner', 'ArgumentPetitioner', 'ArgumentRespondent', 'Fact']
Validation passed.


In [70]:
from pathlib import Path
import zipfile
import json

prediction_path = Path(
    "/kaggle/working/phase3_m3_lorma/prediction.json"
)
zip_path = Path(
    "/kaggle/working/phase3_m3_lorma/submission_M3.zip"
)

# Confirm the JSON is readable before packaging.
with open(prediction_path, "r", encoding="utf-8") as f:
    predictions = json.load(f)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname="prediction.json")

with zipfile.ZipFile(zip_path, "r") as zf:
    print("ZIP contents:", zf.namelist())
    print("ZIP integrity:", zf.testzip())

print("Documents:", len(predictions))
print("Submission ZIP:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 3))

ZIP contents: ['prediction.json']
ZIP integrity: None
Documents: 10
Submission ZIP: /kaggle/working/phase3_m3_lorma/submission_M3.zip
ZIP size (MB): 0.002


PHASE 4

In [2]:
import os
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev": "Phase_4_CJPE/Train/dev.json",
    "test": "Phase_4_CJPE/Test/test.json",
}
phase4_paths = {}
for split, filename in PHASE4_FILES.items():
    print(f"Accessing {split}: {filename}")
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    phase4_paths[split] = path
    print("Cached path:", path)

with open(phase4_paths["train"], "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(phase4_paths["dev"], "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(phase4_paths["test"], "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nDataset sizes:")
print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("\nFirst training example keys:", train_data[0].keys())
print("First test example keys:", test_data[0].keys())

Accessing train: Phase_4_CJPE/Train/train.json


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
Accessing dev: Phase_4_CJPE/Train/dev.json


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
Accessing test: Phase_4_CJPE/Test/test.json


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Cached path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json

Dataset sizes:
Train: 37387
Dev: 1988
Test: 1517

First training example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])
First test example keys: dict_keys(['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5'])


In [11]:
import os
import pickle
import torch
from tqdm.auto import tqdm

OUT = "/kaggle/working/cjpe_m3"
os.makedirs(OUT, exist_ok=True)

with open("/kaggle/input/datasets/spikie/capstone-split/legal_bpe_32k.pkl","rb") as f:
    tokenizer = pickle.load(f)

def encode_rows(rows, name, deduplicate=False):
    seen = set()
    ids_list, labels, case_ids = [], [], []

    for row in tqdm(rows, desc=name):
        if deduplicate and row["id"] in seen:
            continue
        seen.add(row["id"])

        ids = tokenizer.encode_text(row["text"])[:1024]
        if len(ids) == 0:
            continue

        ids_list.append(torch.tensor(ids, dtype=torch.long))
        labels.append(int(row["label"]))
        case_ids.append(row["id"])

    return {
        "ids": ids_list,
        "labels": torch.tensor(labels, dtype=torch.long),
        "case_ids": case_ids
    }

train_enc = encode_rows(train_data, "Tokenizing train", True)
dev_enc = encode_rows(dev_data, "Tokenizing dev")

torch.save(train_enc, OUT + "/train.pt")
torch.save(dev_enc, OUT + "/dev.pt")

print("Train:", len(train_enc["ids"]))
print("Dev:", len(dev_enc["ids"]))
print("Saved tokenized data to:", OUT)

Tokenizing train:   0%|          | 0/37387 [00:00<?, ?it/s]

Tokenizing dev:   0%|          | 0/1988 [00:00<?, ?it/s]

Train: 32305
Dev: 1988
Saved tokenized data to: /kaggle/working/cjpe_m3


In [15]:

%%writefile /kaggle/working/train_m3_cjpe.py
import os
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from sklearn.metrics import f1_score, precision_score, recall_score

from m3_ddp_test import M3Model


# ---------------- CONFIG ----------------
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 2
EPOCHS = 3
LR = 1e-4
NUM_WORKERS = 2
RANK_ADAPTER = 8

PRETRAINED = "/kaggle/working/m3_checkpoints/latest.pt"
TRAIN_FILE = "/kaggle/working/cjpe_m3/train.pt"
DEV_FILE = "/kaggle/working/cjpe_m3/dev.pt"
OUT_DIR = "/kaggle/working/cjpe_m3_results"

os.makedirs(OUT_DIR, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class CJDataset(Dataset):
    def __init__(self, path):
        self.data = torch.load(
            path, map_location="cpu", weights_only=False
        )

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        ids = self.data["ids"][idx][:MAX_LEN]
        label = int(self.data["labels"][idx])
        return torch.as_tensor(ids, dtype=torch.long), label


def collate_fn(batch):
    ids = [item[0] for item in batch]
    labels = torch.tensor(
        [item[1] for item in batch], dtype=torch.long
    )
    lengths = torch.tensor([len(x) for x in ids])

    input_ids = nn.utils.rnn.pad_sequence(
        ids, batch_first=True, padding_value=0
    )

    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)

    return input_ids, mask, labels


class LoRALinear(nn.Module):
    """Frozen linear layer plus a trainable rank-r update."""

    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError("Adapter target is not nn.Linear.")

        self.base = base
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        self.rank = rank
        self.scale = alpha / rank
        self.A = nn.Parameter(
            torch.empty(rank, base.in_features)
        )
        self.B = nn.Parameter(
            torch.zeros(base.out_features, rank)
        )
        nn.init.kaiming_uniform_(self.A, a=np.sqrt(5))

    def forward(self, x):
        update = (x @ self.A.T) @ self.B.T
        return self.base(x) + self.scale * update


def install_adapters(model):
    # Freeze all Phase 1 parameters.
    for p in model.parameters():
        p.requires_grad = False

    targets = []
    for module_name, module in model.named_modules():
        if module_name.endswith("q_proj") or module_name.endswith("o_proj"):
            targets.append(module_name)

    if not targets:
        raise RuntimeError(
            "No q_proj/o_proj modules found. Inspect the actual "
            "M3 attention module names before training."
        )

    for full_name in targets:
        parent_name, child_name = full_name.rsplit(".", 1)
        parent = model.get_submodule(parent_name)
        old_layer = getattr(parent, child_name)
        setattr(
            parent, child_name,
            LoRALinear(old_layer, rank=RANK_ADAPTER)
        )

    print("Installed adapters on:")
    for name in targets:
        print(" ", name)

    return targets


class M3Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, mask):
        x = self.backbone.tok_emb(input_ids)

        for block in self.backbone.blocks:
            x = block(x)

        hidden = self.backbone.ln_f(x)
        # hidden: [batch, sequence, 768]

        mask_float = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask_float).sum(dim=1)
        pooled = pooled / mask_float.sum(dim=1).clamp(min=1.0)

        return self.classifier(pooled)


def main():
    set_seed(SEED)

    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")
    device = torch.device("cuda", local_rank)

    try:
        train_ds = CJDataset(TRAIN_FILE)
        dev_ds = CJDataset(DEV_FILE)

        train_sampler = DistributedSampler(
            train_ds, num_replicas=world_size,
            rank=rank, shuffle=True, seed=SEED
        )

        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE,
            sampler=train_sampler, num_workers=NUM_WORKERS,
            pin_memory=True, collate_fn=collate_fn,
            persistent_workers=(NUM_WORKERS > 0)
        )

        dev_loader = None
        if rank == 0:
            dev_loader = DataLoader(
                dev_ds, batch_size=4, shuffle=False,
                num_workers=NUM_WORKERS, pin_memory=True,
                collate_fn=collate_fn,
                persistent_workers=(NUM_WORKERS > 0)
            )

        model = M3Model()
        ckpt = torch.load(
            PRETRAINED, map_location="cpu", weights_only=False
        )
        state = ckpt["model"] if "model" in ckpt else ckpt
        model.load_state_dict(state, strict=True)

        # Attach adapters AFTER loading Phase 1 weights.
        install_adapters(model)

        model = M3Classifier(model).to(device)

        trainable = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )
        if rank == 0:
            print("Train/dev:", len(train_ds), len(dev_ds))
            print("GPUs:", world_size)
            print("Sequence length:", MAX_LEN)
            print("Trainable parameters:", trainable)

        model = DDP(
            model, device_ids=[local_rank],
            output_device=local_rank
        )

        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR, weight_decay=0.01
        )
        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler("cuda")

        # -------- Smoke test --------
        model.train()
        batch = next(iter(train_loader))
        x, mask, y = [t.to(device) for t in batch]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x, mask)
            loss = criterion(logits, y)

        if logits.shape != (len(y), 2):
            raise RuntimeError("Incorrect classifier output shape.")
        if not torch.isfinite(loss).item():
            raise RuntimeError("Non-finite smoke-test loss.")

        scaler.scale(loss).backward()

        has_grad = False
        for p in model.parameters():
            if p.requires_grad and p.grad is not None:
                if not torch.isfinite(p.grad).all().item():
                    raise RuntimeError("Non-finite gradient.")
                has_grad = True

        if not has_grad:
            raise RuntimeError("No trainable gradients.")

        optimizer.zero_grad(set_to_none=True)
        del x, mask, y, logits, loss, batch
        torch.cuda.empty_cache()
        dist.barrier()

        if rank == 0:
            print("SMOKE TEST PASSED.", flush=True)

        best_f1 = -1.0

        for epoch in range(EPOCHS):
            train_sampler.set_epoch(epoch)
            model.train()
            total_loss = 0.0
            steps = 0

            progress = tqdm(
                train_loader, desc="Epoch {}/{}".format(
                    epoch + 1, EPOCHS
                ), disable=(rank != 0)
            )

            for x, mask, y in progress:
                x = x.to(device, non_blocking=True)
                mask = mask.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(x, mask)
                    loss = criterion(logits, y)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    1.0
                )
                scaler.step(optimizer)
                scaler.update()

                total_loss += loss.item()
                steps += 1

                if rank == 0:
                    progress.set_postfix(loss=round(loss.item(), 4))

            dist.barrier()

            if rank == 0:
                model.eval()
                all_preds = []
                all_labels = []
                dev_loss = 0.0
                dev_steps = 0

                with torch.inference_mode():
                    for x, mask, y in tqdm(
                        dev_loader, desc="Validation"
                    ):
                        x = x.to(device, non_blocking=True)
                        mask = mask.to(device, non_blocking=True)
                        y = y.to(device, non_blocking=True)

                        with torch.autocast("cuda", dtype=torch.float16):
                            logits = model.module(x, mask)
                            loss = criterion(logits, y)

                        dev_loss += loss.item()
                        dev_steps += 1
                        all_preds.extend(
                            logits.argmax(1).cpu().tolist()
                        )
                        all_labels.extend(y.cpu().tolist())

                macro_f1 = f1_score(
                    all_labels, all_preds, average="macro",
                    zero_division=0
                )
                precision = precision_score(
                    all_labels, all_preds, zero_division=0
                )
                recall = recall_score(
                    all_labels, all_preds, zero_division=0
                )

                print(
                    "Epoch {} | train_loss={:.4f} | dev_loss={:.4f} "
                    "| macro_f1={:.4f} | precision={:.4f} | recall={:.4f}"
                    .format(
                        epoch + 1, total_loss / max(steps, 1),
                        dev_loss / max(dev_steps, 1),
                        macro_f1, precision, recall
                    ),
                    flush=True
                )

                saved = {
                    "epoch": epoch + 1,
                    "model": model.module.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "macro_f1": macro_f1
                }
                torch.save(saved, OUT_DIR + "/latest.pt")

                if macro_f1 > best_f1:
                    best_f1 = macro_f1
                    torch.save(saved, OUT_DIR + "/best.pt")
                    print("Saved best checkpoint.", flush=True)

            dist.barrier()

    finally:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


Writing /kaggle/working/train_m3_cjpe.py


In [16]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m3_cjpe.py

W1021 17:06:42.371000 188 torch/distributed/run.py:851] 
W1021 17:06:42.371000 188 torch/distributed/run.py:851] *****************************************
W1021 17:06:42.371000 188 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1021 17:06:42.371000 188 torch/distributed/run.py:851] *****************************************
Installed adapters on:
  blocks.0.attn.q_proj
  blocks.0.attn.o_proj
  blocks.1.attn.q_proj
  blocks.1.attn.o_proj
  blocks.2.attn.q_proj
  blocks.2.attn.o_proj
  blocks.3.attn.q_proj
  blocks.3.attn.o_proj
  blocks.4.attn.q_proj
  blocks.4.attn.o_proj
  blocks.5.attn.q_proj
  blocks.5.attn.o_proj
  blocks.6.attn.q_proj
  blocks.6.attn.o_proj
  blocks.7.attn.q_proj
  blocks.7.attn.o_proj
  blocks.8.attn.q_proj
  blocks.8.attn.o_proj
  blocks.9.attn.q_proj
  blocks.9.attn.o_pro

In [22]:

import os
import re

SOURCE = "/kaggle/working/train_m3_cjpe.py"
TARGET = "/kaggle/working/train_m3_cjpe_lorma.py"
OUT_DIR = "/kaggle/working/cjpe_m3_lorma_results"

assert os.path.exists(SOURCE), f"Missing source script: {SOURCE}"

with open(SOURCE, "r") as f:
    script = f.read()

# Replace the ordinary LoRA layer with the implementation from
# the Phase 3 notebook.
lorma_class = r'''
class LoRMAPlusLinear(nn.Module):
    """LoRMA+ adapter: low-rank update of the frozen base output."""

    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError("Adapter target is not nn.Linear.")

        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.scale = alpha / rank

        # Match the Phase 3 notebook's LoRMAPlusLinear implementation.
        out_features = base.out_features
        self.A = nn.Parameter(torch.empty(rank, out_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.normal_(self.A, mean=0.0, std=0.02)

    def forward(self, x):
        base_output = self.base(x)
        adaptation = (base_output @ self.A.T) @ self.B.T
        return base_output + self.scale * adaptation


def install_adapters(model):
'''

pattern = r"class LoRALinear\(nn\.Module\):.*?\n\ndef install_adapters\(model\):"
script, n = re.subn(pattern, lorma_class, script, count=1, flags=re.S)
assert n == 1, "Could not safely replace LoRALinear implementation."

script, n = re.subn(
    r"LoRALinear\(old_layer,\s*rank=RANK_ADAPTER\)",
    "LoRMAPlusLinear(old_layer, rank=RANK_ADAPTER)",
    script
)
assert n >= 1, "No adapter installation call was replaced."

# Use a separate output folder; preserve all ordinary-LoRA results.
script, n = re.subn(
    r'OUT_DIR\s*=\s*"/kaggle/working/cjpe_m3_results"',
    f'OUT_DIR = "{OUT_DIR}"',
    script,
    count=1
)
assert n == 1, "Could not safely change output directory."

# Explicitly keep the original Phase 1 initialization and data.
assert 'PRETRAINED = "/kaggle/working/m3_checkpoints/latest.pt"' in script
assert 'TRAIN_FILE = "/kaggle/working/cjpe_m3/train.pt"' in script
assert 'DEV_FILE = "/kaggle/working/cjpe_m3/dev.pt"' in script

os.makedirs(OUT_DIR, exist_ok=True)
with open(TARGET, "w") as f:
    f.write(script)

print("Created:", TARGET)
print("Output directory:", OUT_DIR)
print("Epochs:", re.search(r"^EPOCHS\s*=\s*(\d+)", script, re.M).group(1))
print("Uses Phase 1 M3 checkpoint: verified")
print("Uses original train/dev data: verified")
print("Ordinary-LoRA checkpoint untouched: verified")
print("\nAdapter class present:", "class LoRMAPlusLinear" in script)
print("Ordinary LoRALinear class remains:", "class LoRALinear" in script)


Created: /kaggle/working/train_m3_cjpe_lorma.py
Output directory: /kaggle/working/cjpe_m3_lorma_results
Epochs: 3
Uses Phase 1 M3 checkpoint: verified
Uses original train/dev data: verified
Ordinary-LoRA checkpoint untouched: verified

Adapter class present: True
Ordinary LoRALinear class remains: False


In [24]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m3_cjpe_lorma.py

W1021 18:15:46.453000 265 torch/distributed/run.py:851] 
W1021 18:15:46.453000 265 torch/distributed/run.py:851] *****************************************
W1021 18:15:46.453000 265 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1021 18:15:46.453000 265 torch/distributed/run.py:851] *****************************************
Installed adapters on:
  blocks.0.attn.q_proj
  blocks.0.attn.o_proj
  blocks.1.attn.q_proj
  blocks.1.attn.o_proj
  blocks.2.attn.q_proj
  blocks.2.attn.o_proj
  blocks.3.attn.q_proj
  blocks.3.attn.o_proj
  blocks.4.attn.q_proj
  blocks.4.attn.o_proj
  blocks.5.attn.q_proj
  blocks.5.attn.o_proj
  blocks.6.attn.q_proj
  blocks.6.attn.o_proj
  blocks.7.attn.q_proj
  blocks.7.attn.o_proj
  blocks.8.attn.q_proj
  blocks.8.attn.o_proj
  blocks.9.attn.q_proj
  blocks.9.attn.o_pro

In [25]:
import torch
from collections import Counter

train = torch.load(
    "/kaggle/working/cjpe_m3/train.pt",
    map_location="cpu",
    weights_only=False
)

dev = torch.load(
    "/kaggle/working/cjpe_m3/dev.pt",
    map_location="cpu",
    weights_only=False
)

print("Train label counts:", Counter(
    torch.as_tensor(train["labels"]).tolist()
))
print("Dev label counts:", Counter(
    torch.as_tensor(dev["labels"]).tolist()
))

Train label counts: Counter({0: 18920, 1: 13385})
Dev label counts: Counter({0: 994, 1: 994})


In [29]:
import os
import torch
import torch.nn as nn
import numpy as np

from torch.utils.data import DataLoader
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from m3_ddp_test import M3Model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 256
RANK_ADAPTER = 8

PRETRAINED = "/kaggle/working/m3_checkpoints/latest.pt"
BEST_PATH = "/kaggle/working/cjpe_m3_lorma_results/best.pt"
DEV_FILE = "/kaggle/working/cjpe_m3/dev.pt"


class CJDataset(torch.utils.data.Dataset):
    def __init__(self, path):
        self.data = torch.load(
            path, map_location="cpu", weights_only=False
        )

    def __len__(self):
        return len(self.data["labels"])

    def __getitem__(self, idx):
        ids = torch.as_tensor(
            self.data["ids"][idx][:MAX_LEN], dtype=torch.long
        )
        return ids, int(self.data["labels"][idx])


def collate_fn(batch):
    ids = [x[0] for x in batch]
    labels = torch.tensor([x[1] for x in batch], dtype=torch.long)
    lengths = torch.tensor([len(x) for x in ids])

    input_ids = nn.utils.rnn.pad_sequence(
        ids, batch_first=True, padding_value=0
    )
    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)

    return input_ids, mask, labels


class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.scale = alpha / rank
        out_features = base.out_features
        self.A = nn.Parameter(torch.empty(rank, out_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.normal_(self.A, mean=0.0, std=0.02)

    def forward(self, x):
        base_output = self.base(x)
        adaptation = (base_output @ self.A.T) @ self.B.T
        return base_output + self.scale * adaptation


def install_adapters(model):
    for p in model.parameters():
        p.requires_grad = False

    targets = []
    for name, module in model.named_modules():
        if name.endswith("q_proj") or name.endswith("o_proj"):
            targets.append(name)

    for name in targets:
        parent_name, child_name = name.rsplit(".", 1)
        parent = model.get_submodule(parent_name)
        old_layer = getattr(parent, child_name)
        setattr(
            parent, child_name,
            LoRMAPlusLinear(old_layer, rank=RANK_ADAPTER)
        )

    if len(targets) != 32:
        raise RuntimeError(
            f"Expected 32 adapter targets, found {len(targets)}"
        )


class M3Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, mask):
        x = self.backbone.tok_emb(input_ids)

        for block in self.backbone.blocks:
            x = block(x)

        hidden = self.backbone.ln_f(x)
        mask_float = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask_float).sum(dim=1)
        pooled = pooled / mask_float.sum(dim=1).clamp(min=1.0)

        return self.classifier(pooled)


# Reconstruct architecture exactly.
backbone = M3Model()
pretrained = torch.load(
    PRETRAINED, map_location="cpu", weights_only=False
)
backbone.load_state_dict(pretrained["model"], strict=True)

install_adapters(backbone)
model = M3Classifier(backbone)

checkpoint = torch.load(
    BEST_PATH, map_location="cpu", weights_only=False
)
model.load_state_dict(checkpoint["model"], strict=True)
model.to(DEVICE)
model.eval()

dev_ds = CJDataset(DEV_FILE)
dev_loader = DataLoader(
    dev_ds, batch_size=16, shuffle=False, collate_fn=collate_fn
)

y_true = []
prob_class1 = []

with torch.inference_mode():
    for input_ids, mask, labels in dev_loader:
        input_ids = input_ids.to(DEVICE)
        mask = mask.to(DEVICE)

        logits = model(input_ids, mask)
        probabilities = torch.softmax(logits.float(), dim=-1)[:, 1]

        y_true.extend(labels.tolist())
        prob_class1.extend(probabilities.cpu().tolist())

y_true = np.asarray(y_true)
prob_class1 = np.asarray(prob_class1)

print("Examples evaluated:", len(y_true))
print("Mean class-1 probability:", prob_class1.mean())
print("Probability percentiles:",
      np.percentile(prob_class1, [0, 10, 25, 50, 75, 90, 100]))

pred_default = (prob_class1 >= 0.5).astype(int)
print("\nDefault threshold = 0.5")
print("Macro-F1:", f1_score(y_true, pred_default, average="macro"))
print("Confusion matrix:\n", confusion_matrix(y_true, pred_default))
print(classification_report(y_true, pred_default, digits=4))

Examples evaluated: 1988
Mean class-1 probability: 0.3153558048715729
Probability percentiles: [0.01834504 0.17686793 0.24797706 0.31709461 0.38667056 0.45677623
 0.66202545]

Default threshold = 0.5
Macro-F1: 0.4155312246553123
Confusion matrix:
 [[976  18]
 [910  84]]
              precision    recall  f1-score   support

           0     0.5175    0.9819    0.6778       994
           1     0.8235    0.0845    0.1533       994

    accuracy                         0.5332      1988
   macro avg     0.6705    0.5332    0.4155      1988
weighted avg     0.6705    0.5332    0.4155      1988



In [30]:
best_threshold = 0.5
best_macro_f1 = -1

for threshold in np.arange(0.05, 0.96, 0.01):
    pred = (prob_class1 >= threshold).astype(int)
    score = f1_score(
        y_true, pred, average="macro", zero_division=0
    )

    if score > best_macro_f1:
        best_macro_f1 = score
        best_threshold = float(threshold)

pred_best = (prob_class1 >= best_threshold).astype(int)

print("Best threshold:", best_threshold)
print("Best dev Macro-F1:", best_macro_f1)
print("Precision:", precision_score(
    y_true, pred_best, zero_division=0
))
print("Recall:", recall_score(
    y_true, pred_best, zero_division=0
))
print("Confusion matrix:\n", confusion_matrix(y_true, pred_best))

Best threshold: 0.34
Best dev Macro-F1: 0.5706886041983403
Precision: 0.5915841584158416
Recall: 0.48088531187122735
Confusion matrix:
 [[664 330]
 [516 478]]


In [38]:
import os
import json
import zipfile
import torch
import torch.nn as nn
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
from m3_ddp_test import M3Model

# ================= CONFIG =================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 256
BATCH_SIZE = 16
THRESHOLD = 0.34
RANK_ADAPTER = 8

PRETRAINED = "/kaggle/working/m3_checkpoints/latest.pt"
CHECKPOINT = "/kaggle/working/cjpe_m3_lorma_results/best.pt"
OUT_JSON = "/kaggle/working/submission_m3_lorma_p4.json"
OUT_ZIP = "/kaggle/working/submission_M3_LoRMA_p4.zip"

# ================= GET ORIGINAL TEST DATA =================
# Reuse test_data if it already exists in the notebook.
if "test_data" in globals():
    raw_test = test_data
else:
    test_path = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename="Phase_4_CJPE/Test/test.json",
        token=True
    )
    with open(test_path, "r", encoding="utf-8") as f:
        raw_test = json.load(f)

if isinstance(raw_test, dict):
    if "data" in raw_test:
        raw_test = raw_test["data"]
    elif "test" in raw_test:
        raw_test = raw_test["test"]
    else:
        raise ValueError(
            f"Unexpected test JSON structure. Keys: {list(raw_test.keys())}"
        )

print("Raw test examples:", len(raw_test))
print("Example keys:", list(raw_test[0].keys()))

# ================= TOKENIZE IN MEMORY =================
# Uses your existing encode_text() function and tokenizer.
assert "encode_text" in globals(), (
    "encode_text() is not defined. Run your tokenizer setup cell first."
)

test_ids = []
test_case_ids = []

for row in tqdm(raw_test, desc="Tokenizing test"):
    case_id = row.get("id")
    if case_id is None:
        raise KeyError("A test record is missing its id.")

    token_ids = encode_text(row["text"])
    test_ids.append(torch.as_tensor(token_ids[:MAX_LEN], dtype=torch.long))
    test_case_ids.append(str(case_id))

assert len(test_ids) == len(test_case_ids)
assert len(set(test_case_ids)) == len(test_case_ids), "Duplicate test IDs!"

class InMemoryTestDataset(Dataset):
    def __len__(self):
        return len(test_ids)

    def __getitem__(self, idx):
        return test_ids[idx], test_case_ids[idx]

def collate_test(batch):
    sequences = [x[0] for x in batch]
    case_ids = [x[1] for x in batch]
    lengths = torch.tensor([len(x) for x in sequences])

    input_ids = nn.utils.rnn.pad_sequence(
        sequences, batch_first=True, padding_value=0
    )
    positions = torch.arange(input_ids.size(1)).unsqueeze(0)
    mask = positions < lengths.unsqueeze(1)

    return input_ids, mask, case_ids

test_loader = DataLoader(
    InMemoryTestDataset(),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_test
)

# ================= LoRMA+ MODEL =================
class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError("Adapter target is not nn.Linear")

        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.scale = alpha / rank
        out_features = base.out_features
        self.A = nn.Parameter(torch.empty(rank, out_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.normal_(self.A, mean=0.0, std=0.02)

    def forward(self, x):
        base_output = self.base(x)
        adaptation = (base_output @ self.A.T) @ self.B.T
        return base_output + self.scale * adaptation

def install_adapters(model):
    for p in model.parameters():
        p.requires_grad = False

    targets = [
        name for name, module in model.named_modules()
        if name.endswith("q_proj") or name.endswith("o_proj")
    ]
    if len(targets) != 32:
        raise RuntimeError(f"Expected 32 adapter targets; found {len(targets)}")

    for name in targets:
        parent_name, child_name = name.rsplit(".", 1)
        parent = model.get_submodule(parent_name)
        old_layer = getattr(parent, child_name)
        setattr(parent, child_name, LoRMAPlusLinear(old_layer, RANK_ADAPTER))

class M3Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, mask):
        x = self.backbone.tok_emb(input_ids)
        for block in self.backbone.blocks:
            x = block(x)

        hidden = self.backbone.ln_f(x)
        mask_float = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask_float).sum(dim=1)
        pooled = pooled / mask_float.sum(dim=1).clamp(min=1.0)
        return self.classifier(pooled)

for path in [PRETRAINED, CHECKPOINT]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

backbone = M3Model()
pretrained = torch.load(PRETRAINED, map_location="cpu", weights_only=False)
backbone.load_state_dict(pretrained["model"], strict=True)

install_adapters(backbone)
model = M3Classifier(backbone)

ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model"], strict=True)
model.to(DEVICE)
model.eval()

print("Loaded checkpoint epoch:", ckpt.get("epoch"))
print("Saved dev Macro-F1:", ckpt.get("macro_f1"))
print("Using threshold:", THRESHOLD)

# ================= GENERATE PREDICTIONS =================
predictions = []

with torch.inference_mode():
    for input_ids, mask, case_ids in tqdm(test_loader, desc="Predicting"):
        input_ids = input_ids.to(DEVICE)
        mask = mask.to(DEVICE)

        logits = model(input_ids, mask)
        probs = torch.softmax(logits.float(), dim=-1)[:, 1]
        preds = (probs >= THRESHOLD).long().cpu().tolist()

        for case_id, pred in zip(case_ids, preds):
            predictions.append({"id": case_id, "pred": int(pred)})

# ================= SAVE + VALIDATE + ZIP =================
assert len(predictions) == len(raw_test)
assert len({x["id"] for x in predictions}) == len(predictions)
assert all(x["pred"] in (0, 1) for x in predictions)

with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT_JSON, arcname="prediction.json")

print("\nSubmission ready!")
print("Examples:", len(predictions))
print("Class counts:", {
    0: sum(x["pred"] == 0 for x in predictions),
    1: sum(x["pred"] == 1 for x in predictions)
})
print("JSON:", OUT_JSON)
print("ZIP:", OUT_ZIP)
print("ZIP size: %.2f KB" % (os.path.getsize(OUT_ZIP) / 1024))
print("Preview:", predictions[:5])

Raw test examples: 1517
Example keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


Tokenizing test:   0%|          | 0/1517 [00:00<?, ?it/s]

Loaded checkpoint epoch: 3
Saved dev Macro-F1: 0.4155312246553123
Using threshold: 0.34


Predicting:   0%|          | 0/95 [00:00<?, ?it/s]


Submission ready!
Examples: 1517
Class counts: {0: 862, 1: 655}
JSON: /kaggle/working/submission_m3_lorma_p4.json
ZIP: /kaggle/working/submission_M3_LoRMA_p4.zip
ZIP size: 6.43 KB
Preview: [{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 1}, {'id': '1991_137', 'pred': 1}, {'id': '1957_125', 'pred': 0}]


Phase 5

In [6]:

import os
import json
from huggingface_hub import hf_hub_download

REPO_ID = "Exploration-Lab/CS-781-Capstone"
DATA_DIR = "/kaggle/working/lsi_m3"
os.makedirs(DATA_DIR, exist_ok=True)

def load_lsi_split(filename):
    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename="Phase_5_LSI/" + filename,
        token=os.environ.get("HF_TOKEN"),
        local_dir=DATA_DIR
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_lsi_split("Train/train.json")
dev_data = load_lsi_split("Train/dev.json")
test_data = load_lsi_split("Test/test.json")

print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("Example fields:", train_data[0].keys())
print("Example label count:", len(train_data[0]["labels"]))

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Train: 42750
Dev: 10181
Test: 13019
Example fields: dict_keys(['id', 'text', 'labels'])
Example label count: 3


In [3]:

import os
import pickle

TOKENIZER_PATH = (
    "/kaggle/input/datasets/spikie/capstone-split/"
    "legal_bpe_32k.pkl"
)

assert os.path.isfile(TOKENIZER_PATH), (
    "Shared tokenizer file not found. Check the attached Kaggle dataset."
)

with open(TOKENIZER_PATH, "rb") as f:
    tok = pickle.load(f)

print("Loaded shared tokenizer:", type(tok).__name__)
print("Vocabulary size:", tok.vocab_size)
print("Test encoding:", tok.encode_text("A legal case", add_eos=True))


Loaded shared tokenizer: ByteLevelBPE
Vocabulary size: 32000
Test encoding: [66, 33, 1320, 33, 422, 0]


In [7]:

import torch
from tqdm.auto import tqdm

WORK = "/kaggle/working/lsi_m3"
CHUNK_SIZE = 1024
OVERLAP = 128
STRIDE = CHUNK_SIZE - OVERLAP
LABEL_IDS = sorted(set(
    int(label)
    for row in train_data
    for label in row["labels"]
))
label_to_idx = {label: i for i, label in enumerate(LABEL_IDS)}

def encode_document(text):
    token_ids = tok.encode_text(text, add_eos=True)
    if len(token_ids) == 0:
        token_ids = [0]

    chunks = []
    for start in range(0, len(token_ids), STRIDE):
        chunk = token_ids[start:start + CHUNK_SIZE]
        if len(chunk) > 0:
            chunks.append(torch.tensor(chunk, dtype=torch.int32))
        if start + CHUNK_SIZE >= len(token_ids):
            break

    return chunks

def encode_split(rows, has_labels, name):
    encoded = []
    for row in tqdm(rows, desc="Tokenizing " + name):
        item = {
            "id": str(row["id"]),
            "chunks": encode_document(row["text"])
        }
        if has_labels:
            item["labels"] = [
                int(label) for label in row["labels"]
            ]
        encoded.append(item)

    output_path = os.path.join(WORK, name + ".pt")
    torch.save(encoded, output_path)
    print("Saved:", output_path)
    return encoded

train_encoded = encode_split(train_data, True, "train")
dev_encoded = encode_split(dev_data, True, "dev")
test_encoded = encode_split(test_data, False, "test")

torch.save(LABEL_IDS, os.path.join(WORK, "label_ids.pt"))

print("Number of labels:", len(LABEL_IDS))
print("Chunks in first training document:", len(train_encoded[0]["chunks"]))
print("Maximum chunks in a training document:",
      max(len(row["chunks"]) for row in train_encoded))


Tokenizing train:   0%|          | 0/42750 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m3/train.pt


Tokenizing dev:   0%|          | 0/10181 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m3/dev.pt


Tokenizing test:   0%|          | 0/13019 [00:00<?, ?it/s]

Saved: /kaggle/working/lsi_m3/test.pt
Number of labels: 100
Chunks in first training document: 4
Maximum chunks in a training document: 232


In [10]:

import torch

path = "/kaggle/working/m3_checkpoints/latest.pt"
checkpoint = torch.load(
    path, map_location="cpu", weights_only=False
)

print("Checkpoint keys:", list(checkpoint.keys()))

if "model" in checkpoint:
    print("Number of model tensors:", len(checkpoint["model"]))
    for key, value in list(checkpoint["model"].items())[:12]:
        print(key, tuple(value.shape))
else:
    print("No 'model' key; inspect the checkpoint structure.")


Checkpoint keys: ['step', 'model', 'optimizer', 'scaler', 'best_val']
Number of model tensors: 180
tok_emb.weight (32000, 768)
blocks.0.ln1.weight (768,)
blocks.0.ln1.bias (768,)
blocks.0.attn.q_proj.weight (768, 768)
blocks.0.attn.k_proj.weight (256, 768)
blocks.0.attn.v_proj.weight (256, 768)
blocks.0.attn.o_proj.weight (768, 768)
blocks.0.ln2.weight (768,)
blocks.0.ln2.bias (768,)
blocks.0.ffn.gate_proj.weight (2304, 768)
blocks.0.ffn.up_proj.weight (2304, 768)
blocks.0.ffn.down_proj.weight (768, 2304)


In [13]:

import os
import torch
import sys

sys.path.insert(0, "/kaggle/working")
from m3_ddp_test import M3Model

candidates = [
    "/kaggle/working/m3_checkpoints/best.pt",
    "/kaggle/working/m3_checkpoints/latest.pt",
]

checkpoint_path = None
for path in candidates:
    if os.path.isfile(path):
        checkpoint_path = path
        break

if checkpoint_path is None:
    raise FileNotFoundError(
        "Could not find best.pt or latest.pt in "
        "/kaggle/working/m3_checkpoints. "
        "Check the checkpoint path from your inspection output."
    )

print("Checkpoint:", checkpoint_path)
print("Size (MB):", round(os.path.getsize(checkpoint_path) / 1024**2, 2))

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

state = checkpoint["model"] if "model" in checkpoint else checkpoint

model = M3Model()
result = model.load_state_dict(state, strict=True)

print("Strict checkpoint load: SUCCESS")
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Embedding shape:", tuple(model.tok_emb.weight.shape))
print("Block count:", len(model.blocks))
print("Checkpoint step:", checkpoint.get("step", "not stored"))
print("Best validation loss:", checkpoint.get("best_val", "not stored"))


Checkpoint: /kaggle/working/m3_checkpoints/best.pt
Size (MB): 514.0
Strict checkpoint load: SUCCESS
Missing keys: []
Unexpected keys: []
Parameters: 134727168
Embedding shape: (32000, 768)
Block count: 16
Checkpoint step: not stored
Best validation loss: not stored


In [ ]:

# ============================================================
# M3 QLoRA — Phase 5 LSI
# Reuses pre-tokenized chunks. No tokenization or JSON loading.
# ============================================================

import os, sys, time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb

from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, accuracy_score

sys.path.insert(0, "/kaggle/working")
from m3_ddp_test import M3Model

# ---------------- CONFIG ----------------
SEED = 42
DEVICE = "cuda"
DATA_DIR = "/kaggle/working/lsi_m3"
PRETRAINED = "/kaggle/working/m3_checkpoints/best.pt"
OUT_DIR = "/kaggle/working/m3_lsi_qlora"
os.makedirs(OUT_DIR, exist_ok=True)

NUM_LABELS = 100
EPOCHS = 1
LR = 1e-4
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
GRAD_ACCUM = 8
LOG_EVERY = 100
SAVE_EVERY = 500
THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))

# ---------------- LOAD PRE-TOKENIZED DATA ----------------
print("Loading existing tokenized tensors...")
train_data = torch.load(
    f"{DATA_DIR}/train.pt", map_location="cpu", weights_only=False
)
dev_data = torch.load(
    f"{DATA_DIR}/dev.pt", map_location="cpu", weights_only=False
)

assert len(train_data) == 42750
assert len(dev_data) == 10181
assert "labels" in train_data[0] and "chunks" in train_data[0]
assert all(0 <= int(y) < NUM_LABELS for row in train_data for y in row["labels"])

print("Train documents:", len(train_data))
print("Dev documents:", len(dev_data))
print("Max chunks/document:", max(len(r["chunks"]) for r in train_data))

# ---------------- LORA ----------------
class LoRALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        base_out = self.base(x)
        update = F.linear(
            F.linear(self.dropout(x).float(), self.A),
            self.B
        )
        return base_out.float() + update * self.scale

def get_parent(root, dotted_name):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]

def quantize_linear(old):
    q = bnb.nn.Linear4bit(
        old.in_features,
        old.out_features,
        bias=(old.bias is not None),
        compute_dtype=torch.float16,
        compress_statistics=True,
        quant_type="nf4"
    )
    q.weight = bnb.nn.Params4bit(
        old.weight.detach().cpu().contiguous(),
        requires_grad=False,
        compress_statistics=True,
        quant_type="nf4"
    )
    if old.bias is not None:
        q.bias = nn.Parameter(
            old.bias.detach().cpu().half(),
            requires_grad=False
        )
    return q

def build_model():
    checkpoint_data = torch.load(
        PRETRAINED, map_location="cpu", weights_only=False
    )
    state = (
        checkpoint_data["model"]
        if isinstance(checkpoint_data, dict)
        and "model" in checkpoint_data
        else checkpoint_data
    )

    backbone = M3Model()
    backbone.load_state_dict(state, strict=True)
    del checkpoint_data, state

    for p in backbone.parameters():
        p.requires_grad = False

    # Quantize linear layers inside transformer blocks only.
    linear_names = [
        name for name, mod in backbone.named_modules()
        if isinstance(mod, nn.Linear) and name.startswith("blocks.")
    ]
    for name in linear_names:
        parent, child = get_parent(backbone, name)
        setattr(parent, child, quantize_linear(getattr(parent, child)))

    # LoRA on attention query and value projections.
    target_names = [
        name for name, mod in backbone.named_modules()
        if name.startswith("blocks.")
        and (name.endswith(".attn.q_proj") or name.endswith(".attn.v_proj"))
        and isinstance(mod, bnb.nn.Linear4bit)
    ]
    for name in target_names:
        parent, child = get_parent(backbone, name)
        setattr(
            parent, child,
            LoRALinear(
                getattr(parent, child),
                rank=LORA_RANK,
                alpha=LORA_ALPHA,
                dropout=LORA_DROPOUT
            )
        )

    # Keep embeddings and norms in FP32, matching the successful smoke test.
    backbone = backbone.to(DEVICE)
    backbone.tok_emb.to(dtype=torch.float32)
    backbone.ln_f.to(dtype=torch.float32)
    for block in backbone.blocks:
        block.ln1.to(dtype=torch.float32)
        block.ln2.to(dtype=torch.float32)

    class LSIClassifier(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            self.classifier = nn.Linear(768, NUM_LABELS)

        def encode_chunk(self, input_ids):
            x = self.backbone.tok_emb(input_ids)
            for block in self.backbone.blocks:
                if self.training and x.requires_grad:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
            x = self.backbone.ln_f(x)
            # Chunk tensors are unpadded; use the last token representation.
            return x[:, -1, :]

        def forward(self, input_ids):
            features = self.encode_chunk(input_ids)
            return self.classifier(features.float())

    model = LSIClassifier(backbone).to(DEVICE)

    for name, p in model.named_parameters():
        p.requires_grad = (
            "lora_A" in name or "lora_B" in name
            or name.startswith("classifier.")
        )

    print("LoRA targets:", len(target_names))
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Trainable parameters:", f"{n_trainable:,}")
    return model

model = build_model()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01
)
scaler = torch.amp.GradScaler("cuda")

# ---------------- CHECKPOINT HELPERS ----------------
RESUME_PATH = f"{OUT_DIR}/last.pt"
BEST_PATH = f"{OUT_DIR}/best.pt"

def adapter_state(model):
    return {
        name: param.detach().cpu().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

def save_checkpoint(path, epoch, step, best_macro):
    torch.save({
        "epoch": epoch,
        "global_step": step,
        "best_macro": best_macro,
        "adapter": adapter_state(model),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "config": {
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "lr": LR,
            "threshold": THRESHOLD
        }
    }, path)

start_epoch = 0
global_step = 0
best_macro = -1.0
resume_batch = 0

if os.path.exists(RESUME_PATH):
    saved = torch.load(
        RESUME_PATH, map_location="cpu", weights_only=False
    )
    named_params = dict(model.named_parameters())
    with torch.no_grad():
        for name, value in saved["adapter"].items():
            named_params[name].copy_(value.to(named_params[name].device))
    optimizer.load_state_dict(saved["optimizer"])
    scaler.load_state_dict(saved["scaler"])
    start_epoch = saved["epoch"]
    global_step = saved["global_step"]
    best_macro = saved["best_macro"]
    resume_batch = saved.get("batch_in_epoch", 0)
    print("Resuming at epoch", start_epoch + 1,
          "batch", resume_batch, "step", global_step)
    del saved

# ---------------- DOCUMENT HELPERS ----------------
def target_tensor(labels):
    y = torch.zeros(NUM_LABELS, dtype=torch.float32, device=DEVICE)
    for label in labels:
        y[int(label)] = 1.0
    return y.unsqueeze(0)

def select_training_chunk(row):
    # Pick a different chunk as the random seed changes each epoch.
    chunks = row["chunks"]
    chunk = chunks[random.randrange(len(chunks))]
    return chunk.long().unsqueeze(0).to(DEVICE)

@torch.no_grad()
def evaluate(model, rows):
    model.eval()
    y_true = []
    y_prob = []
    loss_sum = 0.0

    for i, row in enumerate(rows):
        # Initial validation protocol: first chunk per document.
        ids = row["chunks"][0].long().unsqueeze(0).to(DEVICE)
        target = target_tensor(row["labels"])

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(ids)
            loss = F.binary_cross_entropy_with_logits(logits, target)

        y_prob.append(torch.sigmoid(logits.float()).cpu().numpy()[0])
        y_true.append(target.cpu().numpy()[0])
        loss_sum += loss.item()

        if (i + 1) % 1000 == 0:
            print(f"Validation {i+1}/{len(rows)}", flush=True)

    y_true = np.asarray(y_true, dtype=np.int32)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= THRESHOLD).astype(np.int32)

    metrics = {
        "loss": loss_sum / len(rows),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "subset_accuracy": accuracy_score(y_true, y_pred)
    }
    model.train()
    return metrics

# ---------------- TRAIN ----------------
for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_start = time.time()
    order = list(range(len(train_data)))
    random.shuffle(order)

    batch_start = resume_batch if epoch == start_epoch else 0
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    for position in range(batch_start, len(order)):
        row = train_data[order[position]]
        ids = select_training_chunk(row)
        target = target_tensor(row["labels"])

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(ids)
            loss = F.binary_cross_entropy_with_logits(logits, target)
            scaled_loss = loss / GRAD_ACCUM

        scaler.scale(scaled_loss).backward()
        running_loss += loss.item()

        should_step = (
            (position + 1) % GRAD_ACCUM == 0
            or position + 1 == len(order)
        )

        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % SAVE_EVERY == 0:
                save_checkpoint(RESUME_PATH, epoch, global_step, best_macro)
                # Record position so the same epoch can resume.
                temp = torch.load(
                    RESUME_PATH, map_location="cpu", weights_only=False
                )
                temp["batch_in_epoch"] = position + 1
                torch.save(temp, RESUME_PATH)
                del temp
                print("Saved resumable checkpoint at step", global_step)

        if (position + 1) % LOG_EVERY == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Document {position+1}/{len(order)} | "
                f"Loss {running_loss/(position+1):.4f} | "
                f"Step {global_step} | "
                f"VRAM {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB",
                flush=True
            )

    resume_batch = 0
    train_loss = running_loss / len(order)
    print(f"\nEpoch {epoch+1} train loss: {train_loss:.4f}")

    metrics = evaluate(model, dev_data)
    print(
        f"DEV | loss={metrics['loss']:.4f} | "
        f"Macro-F1={metrics['macro_f1']:.5f} | "
        f"Micro-F1={metrics['micro_f1']:.5f} | "
        f"Subset accuracy={metrics['subset_accuracy']:.5f}",
        flush=True
    )

    if metrics["macro_f1"] > best_macro:
        best_macro = metrics["macro_f1"]
        save_checkpoint(BEST_PATH, epoch + 1, global_step, best_macro)
        print("Saved new best adapter checkpoint:", BEST_PATH)

    save_checkpoint(RESUME_PATH, epoch + 1, global_step, best_macro)
    print(
        f"Epoch time: {(time.time()-epoch_start)/60:.1f} min | "
        f"Best Macro-F1: {best_macro:.5f}",
        flush=True
    )

print("\nTraining finished.")
print("Best adapter:", BEST_PATH)
print("Resume checkpoint:", RESUME_PATH)


GPU: Tesla T4
Loading existing tokenized tensors...
Train documents: 42750
Dev documents: 10181
Max chunks/document: 232
LoRA targets: 32
Trainable parameters: 76,900
Epoch 1/1 | Document 100/42750 | Loss 0.6792 | Step 12 | VRAM 1.03 GiB
Epoch 1/1 | Document 200/42750 | Loss 0.5914 | Step 25 | VRAM 1.03 GiB
Epoch 1/1 | Document 300/42750 | Loss 0.5352 | Step 37 | VRAM 1.03 GiB
Epoch 1/1 | Document 400/42750 | Loss 0.4905 | Step 50 | VRAM 1.03 GiB
Epoch 1/1 | Document 500/42750 | Loss 0.4489 | Step 62 | VRAM 1.03 GiB
Epoch 1/1 | Document 600/42750 | Loss 0.4217 | Step 75 | VRAM 1.03 GiB
Epoch 1/1 | Document 700/42750 | Loss 0.3949 | Step 87 | VRAM 1.03 GiB
Epoch 1/1 | Document 800/42750 | Loss 0.3733 | Step 100 | VRAM 1.03 GiB
Epoch 1/1 | Document 900/42750 | Loss 0.3585 | Step 112 | VRAM 1.03 GiB
Epoch 1/1 | Document 1000/42750 | Loss 0.3461 | Step 125 | VRAM 1.03 GiB
Epoch 1/1 | Document 1100/42750 | Loss 0.3333 | Step 137 | VRAM 1.03 GiB
Epoch 1/1 | Document 1200/42750 | Loss 0.3214 

In [7]:
import os
import sys
import json
import zipfile
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb
import numpy as np

from torch.utils.checkpoint import checkpoint

# ---------------- PATHS ----------------
sys.path.insert(0, "/kaggle/working")
from m3_ddp_test import M3Model

DATA_DIR = "/kaggle/working/lsi_m3"
PRETRAINED = "/kaggle/working/m3_checkpoints/best.pt"
TEST_PATH = os.path.join(DATA_DIR, "test.pt")
LABEL_PATH = os.path.join(DATA_DIR, "label_ids.pt")
OUT_DIR = "/kaggle/working/m3_lsi_qlora"

BEST_PATH = os.path.join(OUT_DIR, "best.pt")
LAST_PATH = os.path.join(OUT_DIR, "last.pt")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_LABELS = 100
THRESHOLD = 0.5
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

assert os.path.exists(PRETRAINED), f"Missing base checkpoint: {PRETRAINED}"
assert os.path.exists(TEST_PATH), f"Missing test data: {TEST_PATH}"
assert os.path.exists(LABEL_PATH), f"Missing label mapping: {LABEL_PATH}"
assert os.path.exists(BEST_PATH), f"Missing adapter: {BEST_PATH}"
assert os.path.exists(LAST_PATH), f"Missing adapter: {LAST_PATH}"

print("Device:", DEVICE)

# ---------------- LOAD TEST DATA ----------------
test_data = torch.load(TEST_PATH, map_location="cpu", weights_only=False)
label_ids = torch.load(LABEL_PATH, map_location="cpu", weights_only=False)

assert len(label_ids) == NUM_LABELS
assert len(test_data) == 13019

# ---------------- MODEL HELPERS ----------------
class LoRALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        self.scale = alpha / rank
        self.dropout = nn.Dropout(dropout)
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        base_out = self.base(x)
        update = F.linear(
            F.linear(self.dropout(x).float(), self.A),
            self.B
        )
        return base_out.float() + update * self.scale


def get_parent(root, dotted_name):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    return parent, parts[-1]


def quantize_linear(old):
    q = bnb.nn.Linear4bit(
        old.in_features,
        old.out_features,
        bias=(old.bias is not None),
        compute_dtype=torch.float16,
        compress_statistics=True,
        quant_type="nf4"
    )
    q.weight = bnb.nn.Params4bit(
        old.weight.detach().cpu().contiguous(),
        requires_grad=False,
        compress_statistics=True,
        quant_type="nf4"
    )
    if old.bias is not None:
        q.bias = nn.Parameter(
            old.bias.detach().cpu().half(),
            requires_grad=False
        )
    return q


class LSIClassifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(768, NUM_LABELS)

    def encode_chunk(self, input_ids):
        x = self.backbone.tok_emb(input_ids)
        for block in self.backbone.blocks:
            x = block(x)
        x = self.backbone.ln_f(x)
        return x[:, -1, :]

    def forward(self, input_ids):
        features = self.encode_chunk(input_ids)
        return self.classifier(features.float())


def build_model():
    saved = torch.load(PRETRAINED, map_location="cpu", weights_only=False)
    base_state = saved["model"] if isinstance(saved, dict) and "model" in saved else saved

    backbone = M3Model()
    backbone.load_state_dict(base_state, strict=True)
    del saved, base_state

    for p in backbone.parameters():
        p.requires_grad = False

    linear_names = [
        name for name, mod in backbone.named_modules()
        if isinstance(mod, nn.Linear) and name.startswith("blocks.")
    ]

    for name in linear_names:
        parent, child = get_parent(backbone, name)
        setattr(parent, child, quantize_linear(getattr(parent, child)))

    target_names = [
        name for name, mod in backbone.named_modules()
        if name.startswith("blocks.")
        and (name.endswith(".attn.q_proj") or name.endswith(".attn.v_proj"))
        and isinstance(mod, bnb.nn.Linear4bit)
    ]

    for name in target_names:
        parent, child = get_parent(backbone, name)
        setattr(
            parent,
            child,
            LoRALinear(
                getattr(parent, child),
                rank=LORA_RANK,
                alpha=LORA_ALPHA,
                dropout=LORA_DROPOUT
            )
        )

    backbone = backbone.to(DEVICE)
    backbone.tok_emb.to(dtype=torch.float32)
    backbone.ln_f.to(dtype=torch.float32)

    for block in backbone.blocks:
        block.ln1.to(dtype=torch.float32)
        block.ln2.to(dtype=torch.float32)

    model = LSIClassifier(backbone).to(DEVICE)

    for name, p in model.named_parameters():
        p.requires_grad = (
            "lora_A" in name or "lora_B" in name
            or name.startswith("classifier.")
        )

    return model


@torch.inference_mode()
def predict_checkpoint(model, checkpoint_path, output_prefix):
    saved = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    adapter = saved["adapter"]
    named_params = dict(model.named_parameters())

    missing = [name for name in adapter if name not in named_params]
    if missing:
        raise KeyError(f"Checkpoint parameter names do not match model: {missing[:5]}")

    with torch.no_grad():
        for name, value in adapter.items():
            named_params[name].copy_(
                value.to(
                    device=named_params[name].device,
                    dtype=named_params[name].dtype
                )
            )

    del saved, adapter
    model.eval()

    predictions = []
    ids = []

    for i, row in enumerate(test_data):
        # Match the validation protocol: first chunk per document.
        input_ids = row["chunks"][0].long().unsqueeze(0).to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE == "cuda")
        ):
            logits = model(input_ids)

        probabilities = torch.sigmoid(logits.float())[0]
        pred_indices = torch.where(probabilities >= THRESHOLD)[0].cpu().tolist()

        # The training labels are integer indices 0..99.
        # label_ids is the ordered mapping used by the training pipeline.
        pred_labels = [int(label_ids[j]) for j in pred_indices]

        ids.append(str(row["id"]))
        predictions.append({
            "id": str(row["id"]),
            "labels": pred_labels
        })

        if (i + 1) % 500 == 0:
            print(f"{output_prefix}: {i+1}/{len(test_data)} documents", flush=True)

    json_path = f"/kaggle/working/predictions_{output_prefix}.json"
    zip_path = f"/kaggle/working/submission_{output_prefix}.zip"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="predictions.json")

    print(f"\nCreated: {json_path}")
    print(f"Created: {zip_path}")
    print("Documents:", len(predictions))
    print("Documents with at least one predicted label:",
          sum(len(x["labels"]) > 0 for x in predictions))
    print("Example:", predictions[0])

    return json_path, zip_path


# ---------------- GENERATE BOTH SUBMISSIONS ----------------
model = build_model()
print("M3 LSI model loaded.")

best_json, best_zip = predict_checkpoint(model, BEST_PATH, "best")
last_json, last_zip = predict_checkpoint(model, LAST_PATH, "last")

print("\n========== FINISHED ==========")
print(best_zip)
print(last_zip)

Device: cuda
M3 LSI model loaded.
best: 500/13019 documents
best: 1000/13019 documents
best: 1500/13019 documents
best: 2000/13019 documents
best: 2500/13019 documents
best: 3000/13019 documents
best: 3500/13019 documents
best: 4000/13019 documents
best: 4500/13019 documents
best: 5000/13019 documents
best: 5500/13019 documents
best: 6000/13019 documents
best: 6500/13019 documents
best: 7000/13019 documents
best: 7500/13019 documents
best: 8000/13019 documents
best: 8500/13019 documents
best: 9000/13019 documents
best: 9500/13019 documents
best: 10000/13019 documents
best: 10500/13019 documents
best: 11000/13019 documents
best: 11500/13019 documents
best: 12000/13019 documents
best: 12500/13019 documents
best: 13000/13019 documents

Created: /kaggle/working/predictions_best.json
Created: /kaggle/working/submission_best.zip
Documents: 13019
Documents with at least one predicted label: 4448
Example: {'id': '100002997', 'labels': []}
last: 500/13019 documents
last: 1000/13019 documents
la

In [6]:
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 29.8 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


Phase 6

In [19]:
import os
import json
import random
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILE = "Phase_6_Summ/Train/train.json"
TEST_FILE = "Phase_6_Summ/Test/test.json"

SEED = 42
random.seed(SEED)


train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training dataset:",train_path)

test_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

Training dataset: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

In [20]:
with open(train_path,"r",encoding="utf-8") as f:
    summ_train_data=json.load(f)

with open(test_path,"r",encoding="utf-8") as f:
    summ_test_data=json.load(f)

random.Random(SEED).shuffle(summ_train_data)

valid_size=703

valid_examples=summ_train_data[:valid_size]
train_examples=summ_train_data[valid_size:]

test_examples=summ_test_data

print("Training documents:",len(train_examples))
print("Validation documents:",len(valid_examples))
print("Test documents:",len(test_examples))

print("\nTrain fields:",train_examples[0].keys())
print("Test fields:",test_examples[0].keys())

print("\nSample ID:",train_examples[0]["id"])
print("Document length:",len(train_examples[0]["document"]))
print("Summary length:",len(train_examples[0]["summary"]))

Training documents: 6327
Validation documents: 703
Test documents: 100

Train fields: dict_keys(['id', 'document', 'summary'])
Test fields: dict_keys(['id', 'document'])

Sample ID: 1812
Document length: 23129
Summary length: 2938


In [29]:
import os
import pickle

TOKENIZER_PATH = ("/kaggle/input/datasets/spikie/capstone-split/""legal_bpe_32k.pkl")
with open(TOKENIZER_PATH, "rb") as f:
    tok = pickle.load(f)

print("Loaded shared tokenizer:", type(tok).__name__)
print("Vocabulary size:", tok.vocab_size)
print("Test encoding:", tok.encode_text("A legal case", add_eos=True))

Loaded shared tokenizer: ByteLevelBPE
Vocabulary size: 32000
Test encoding: [66, 33, 1320, 33, 422, 0]


In [22]:
import os
import torch
from tqdm.auto import tqdm

DATA_DIR = "/kaggle/working/phase6_m3_data"
os.makedirs(DATA_DIR,exist_ok=True)

CHUNK_SIZE = 1024
CHUNK_OVERLAP = 128
MAX_TARGET_LEN = 1023
EOS_ID = tok.eos_id

def make_chunks(document):
    ids=tok.encode_text(document,add_eos=False)
    chunks=[]
    step=CHUNK_SIZE-CHUNK_OVERLAP
    for start in range(0,len(ids),step):
        chunk=ids[start:start+CHUNK_SIZE]
        if len(chunk)==0:
            continue

        chunks.append(chunk)
        if start+CHUNK_SIZE>=len(ids):
            break

    return chunks


def prepare_train_data(examples,desc):
    processed=[]
    for example in tqdm(examples,desc=desc):
        source_chunks=make_chunks(example["document"])
        target_ids=tok.encode_text(
            example["summary"],
            add_eos=False
        )

        target_ids=target_ids[:MAX_TARGET_LEN-1]
        target_ids.append(EOS_ID)
        for chunk in source_chunks:
            processed.append({
                "id":example["id"],
                "source_ids":chunk,
                "target_ids":target_ids})

    return processed


def prepare_test_data(examples):
    processed=[]
    for example in tqdm(examples,desc="Processing test documents"):
        source_chunks=make_chunks(example["document"])
        for i,chunk in enumerate(source_chunks):
            processed.append({
                "id":example["id"],
                "chunk_idx":i,
                "source_ids":chunk
            })

    return processed


train_chunks=prepare_train_data(train_examples,"Processing training documents")
valid_chunks=prepare_train_data(valid_examples,"Processing validation documents")
test_chunks=prepare_test_data(test_examples)

torch.save(train_chunks,f"{DATA_DIR}/train.pt")
torch.save(valid_chunks,f"{DATA_DIR}/valid.pt")
torch.save(test_chunks,f"{DATA_DIR}/test.pt")

print("\nProcessed data:")
print("Training examples:",len(train_chunks))
print("Validation examples:",len(valid_chunks))
print("Test chunks:",len(test_chunks))

print("\nSample:")
print("ID:",train_chunks[0]["id"])
print("Source length:",len(train_chunks[0]["source_ids"]))
print("Target length:",len(train_chunks[0]["target_ids"]))
print("Maximum sequence length:",
      max(len(x["source_ids"])+len(x["target_ids"]) for x in train_chunks))

print("\nSaved to:",DATA_DIR)

Processing training documents:   0%|          | 0/6327 [00:00<?, ?it/s]

Processing validation documents:   0%|          | 0/703 [00:00<?, ?it/s]

Processing test documents:   0%|          | 0/100 [00:00<?, ?it/s]


Processed data:
Training examples: 66782
Validation examples: 7308
Test chunks: 1152

Sample:
ID: 1812
Source length: 1024
Target length: 1023
Maximum sequence length: 2047

Saved to: /kaggle/working/phase6_m3_data


In [23]:
CKPT_PATH = "/kaggle/working/m3_checkpoints/best.pt"

model = M3Model()

state_dict = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=True
)

result = model.load_state_dict(state_dict,strict=True)

print("M3 checkpoint loaded successfully")
print("Missing keys:",result.missing_keys)
print("Unexpected keys:",result.unexpected_keys)
print("Parameters:",sum(p.numel() for p in model.parameters()))

M3 checkpoint loaded successfully
Missing keys: []
Unexpected keys: []
Parameters: 134727168


In [24]:
import types

PREFIX_LEN = 32

def apply_rope_positions(x,positions):
    head_dim=x.size(-1)

    inv_freq=1.0/(10000**(
        torch.arange(0,head_dim,2,device=x.device,dtype=torch.float32)/head_dim
    ))

    freqs=torch.outer(positions.float(),inv_freq)
    cos=freqs.cos().to(x.dtype)[None,None,:,:]
    sin=freqs.sin().to(x.dtype)[None,None,:,:]

    x1=x[...,::2]
    x2=x[...,1::2]

    return torch.stack(
        [x1*cos-x2*sin,x1*sin+x2*cos],
        dim=-1
    ).flatten(-2)


def prefix_attention_forward(self,x):
    B,T,C=x.shape
    P=self.prefix_len

    q=self.q_proj(x)
    k=self.k_proj(x)
    v=self.v_proj(x)

    q=q.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
    k=k.view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)
    v=v.view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)

    positions=torch.arange(P,P+T,device=x.device)
    prefix_positions=torch.arange(P,device=x.device)

    q=apply_rope_positions(q,positions)
    k=apply_rope_positions(k,positions)

    prefix_k=self.prefix_k.expand(B,-1,-1,-1)
    prefix_v=self.prefix_v.expand(B,-1,-1,-1)

    prefix_k=apply_rope_positions(prefix_k,prefix_positions)

    k=torch.cat([prefix_k,k],dim=2)
    v=torch.cat([prefix_v,v],dim=2)

    repeat_factor=self.n_heads//self.n_kv_heads

    k=k.repeat_interleave(repeat_factor,dim=1)
    v=v.repeat_interleave(repeat_factor,dim=1)

    query_pos=torch.arange(T,device=x.device)
    key_pos=torch.arange(T,device=x.device)

    content_mask=(
        (key_pos[None,:]<=query_pos[:,None]) &
        (key_pos[None,:]>=(query_pos[:,None]-self.window_size+1))
    )

    prefix_mask=torch.ones(T,P,device=x.device,dtype=torch.bool)
    mask=torch.cat([prefix_mask,content_mask],dim=1)
    mask=mask.unsqueeze(0).unsqueeze(0)

    y=F.scaled_dot_product_attention(
        q,k,v,
        attn_mask=mask,
        dropout_p=self.dropout if self.training else 0.0
    )

    y=y.transpose(1,2).contiguous().view(
        B,T,self.n_heads*self.head_dim
    )

    return self.o_proj(y)


for block in model.blocks:
    attn=block.attn

    attn.prefix_len=PREFIX_LEN

    attn.prefix_k=nn.Parameter(
        torch.empty(1,attn.n_kv_heads,PREFIX_LEN,attn.head_dim)
    )

    attn.prefix_v=nn.Parameter(
        torch.empty(1,attn.n_kv_heads,PREFIX_LEN,attn.head_dim)
    )

    nn.init.normal_(attn.prefix_k,mean=0.0,std=0.02)
    nn.init.normal_(attn.prefix_v,mean=0.0,std=0.02)

    attn.forward=types.MethodType(prefix_attention_forward,attn)


for p in model.parameters():
    p.requires_grad=False

for block in model.blocks:
    block.attn.prefix_k.requires_grad=True
    block.attn.prefix_v.requires_grad=True


total_params=sum(p.numel() for p in model.parameters())
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Prefix tuning enabled")
print("Prefix length:",PREFIX_LEN)
print("Total parameters:",total_params)
print("Trainable parameters:",trainable_params)
print("Trainable percentage:",round(100*trainable_params/total_params,4),"%")

Prefix tuning enabled
Prefix length: 32
Total parameters: 134989312
Trainable parameters: 262144
Trainable percentage: 0.1942 %


In [25]:
def prefix_attention_forward(self,x):
    B,T,C=x.shape
    P=self.prefix_len

    q=self.q_proj(x)
    k=self.k_proj(x)
    v=self.v_proj(x)

    q=q.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
    k=k.view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)
    v=v.view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)

    positions=torch.arange(P,P+T,device=x.device)
    prefix_positions=torch.arange(P,device=x.device)

    q=apply_rope_positions(q,positions)
    k=apply_rope_positions(k,positions)

    prefix_k=self.prefix_k.to(dtype=k.dtype).expand(B,-1,-1,-1)
    prefix_v=self.prefix_v.to(dtype=v.dtype).expand(B,-1,-1,-1)

    prefix_k=apply_rope_positions(prefix_k,prefix_positions)

    k=torch.cat([prefix_k,k],dim=2)
    v=torch.cat([prefix_v,v],dim=2)

    repeat_factor=self.n_heads//self.n_kv_heads

    k=k.repeat_interleave(repeat_factor,dim=1)
    v=v.repeat_interleave(repeat_factor,dim=1)

    query_pos=torch.arange(T,device=x.device)
    key_pos=torch.arange(T,device=x.device)

    content_mask=(
        (key_pos[None,:]<=query_pos[:,None]) &
        (key_pos[None,:]>=(query_pos[:,None]-self.window_size+1))
    )

    prefix_mask=torch.ones(T,P,device=x.device,dtype=torch.bool)
    mask=torch.cat([prefix_mask,content_mask],dim=1)
    mask=mask.unsqueeze(0).unsqueeze(0)

    y=F.scaled_dot_product_attention(
        q,k,v,
        attn_mask=mask,
        dropout_p=self.dropout if self.training else 0.0
    )

    y=y.transpose(1,2).contiguous().view(
        B,T,self.n_heads*self.head_dim
    )

    return self.o_proj(y)


for block in model.blocks:
    block.attn.forward=types.MethodType(
        prefix_attention_forward,block.attn
    )

print("Prefix attention updated for mixed precision.")

Prefix attention updated for mixed precision.


In [27]:
%%writefile /kaggle/working/train_m3_phase6.py
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
os.environ.setdefault("OMP_NUM_THREADS","1")

import math
import types
import argparse
import random
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

from torch.utils.data import Dataset,DataLoader,Subset
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast,GradScaler
from torch.utils.checkpoint import checkpoint
from torch.nn.attention import sdpa_kernel,SDPBackend
from tqdm import tqdm

from m3_ddp_test import M3Model,VOCAB_SIZE


# ================= CONFIG =================

PRETRAINED="/kaggle/working/m3_checkpoints/best.pt"
DATA_DIR="/kaggle/working/phase6_m3_data"
CKPT_DIR="/kaggle/working/phase6_m3_prefix_2chunks"

TRAIN_PATH=f"{DATA_DIR}/train.pt"
VALID_PATH=f"{DATA_DIR}/valid.pt"

PREFIX_LEN=32
BLOCK_SIZE=2048
BATCH_SIZE=1
GRAD_ACCUM=8
NUM_WORKERS=2

EPOCHS=3
LR=5e-4
MIN_LR=5e-5
WEIGHT_DECAY=0.01
GRAD_CLIP=1.0
SAVE_EVERY=500

TRAIN_CHUNKS_PER_DOC=2
VALID_CHUNKS_PER_DOC=2

SMOKE_BATCHES=2
SMOKE_SOURCE_LEN=128
SMOKE_TARGET_LEN=64
SMOKE_CKPT="/kaggle/working/m3_phase6_smoke_test.pt"

os.makedirs(CKPT_DIR,exist_ok=True)


# ================= PREFIX ATTENTION =================

def prefix_attention_forward(self,x):
    B,T,C=x.shape
    P=self.prefix_len

    if T>BLOCK_SIZE:
        raise RuntimeError(f"Input length {T} exceeds BLOCK_SIZE={BLOCK_SIZE}")

    q=self.q_proj(x)
    k=self.k_proj(x)
    v=self.v_proj(x)

    q=q.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
    k=k.view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)
    v=v.view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)

    positions=torch.arange(T,device=x.device)+P

    def apply_rope(z,pos):
        cos=self.prefix_rope_cos.index_select(0,pos).to(z.dtype)[None,None,:,:]
        sin=self.prefix_rope_sin.index_select(0,pos).to(z.dtype)[None,None,:,:]

        z1=z[...,::2]
        z2=z[...,1::2]

        return torch.stack(
            [z1*cos-z2*sin,z1*sin+z2*cos],
            dim=-1
        ).flatten(-2)

    q=apply_rope(q,positions)
    k=apply_rope(k,positions)

    prefix_positions=torch.arange(P,device=x.device)

    # Keep prefix parameters in FP32, cast for attention computation.
    prefix_k=apply_rope(self.prefix_k.to(k.dtype),prefix_positions)
    prefix_v=self.prefix_v.to(v.dtype)

    k=torch.cat(
        [prefix_k.expand(B,-1,-1,-1),k],
        dim=-2
    )
    v=torch.cat(
        [prefix_v.expand(B,-1,-1,-1),v],
        dim=-2
    )

    mask=self.prefix_mask[:,:,:T,:P+T]

    # T4 efficient attention requires matching Q/K/V head counts.
    repeat_factor=q.size(1)//k.size(1)
    k=k.repeat_interleave(repeat_factor,dim=1)
    v=v.repeat_interleave(repeat_factor,dim=1)

    with sdpa_kernel(backends=[SDPBackend.EFFICIENT_ATTENTION]):
        y=F.scaled_dot_product_attention(
            q,k,v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0
        )

    y=y.transpose(1,2).contiguous().view(
        B,T,self.n_heads*self.head_dim
    )

    return self.o_proj(y)


def install_prefixes(model):
    for p in model.parameters():
        p.requires_grad=False

    P=PREFIX_LEN
    TMAX=BLOCK_SIZE
    device=next(model.parameters()).device
    first_attn=model.blocks[0].attn

    qpos=torch.arange(TMAX,device=device)[:,None]
    kpos=torch.arange(TMAX,device=device)[None,:]

    local_mask=(
        (kpos<=qpos) &
        (kpos>=qpos-first_attn.window_size+1)
    )

    prefix_mask=torch.cat(
        [
            torch.ones(TMAX,P,dtype=torch.bool,device=device),
            local_mask
        ],
        dim=-1
    )[None,None,:,:]

    inv_freq=1.0/(
        10000**(
            torch.arange(
                0,first_attn.head_dim,2,
                dtype=torch.float32,
                device=device
            )/first_attn.head_dim
        )
    )

    pos=torch.arange(P+TMAX,dtype=torch.float32,device=device)
    freqs=torch.outer(pos,inv_freq)

    rope_cos=freqs.cos()
    rope_sin=freqs.sin()

    for block in model.blocks:
        attn=block.attn
        attn.prefix_len=P

        attn.prefix_k=nn.Parameter(
            torch.randn(
                1,attn.n_kv_heads,P,attn.head_dim,
                device=device
            )*0.02
        )

        attn.prefix_v=nn.Parameter(
            torch.randn(
                1,attn.n_kv_heads,P,attn.head_dim,
                device=device
            )*0.02
        )

        attn.register_buffer(
            "prefix_mask",prefix_mask.clone(),persistent=False
        )
        attn.register_buffer(
            "prefix_rope_cos",rope_cos.clone(),persistent=False
        )
        attn.register_buffer(
            "prefix_rope_sin",rope_sin.clone(),persistent=False
        )

        attn.forward=types.MethodType(prefix_attention_forward,attn)

    return model


def checkpointed_forward(self,idx,target_start):
    x=self.tok_emb(idx)

    for block in self.blocks:
        if self.training:
            x=checkpoint(block,x,use_reentrant=False)
        else:
            x=block(x)

    x=self.ln_f(x)

    # Source positions do not contribute to the loss.
    # Only calculate vocabulary logits for summary positions.
    x=x[:,target_start:,:]

    return F.linear(x,self.tok_emb.weight)


# ================= DATA =================

class SummaryDataset(Dataset):
    def __init__(self,path):
        self.data=torch.load(path,map_location="cpu",weights_only=False)
        self.doc_groups={}

        for idx,item in enumerate(self.data):
            doc_id=item.get("id",item.get("doc_id",item.get("document_id")))

            if doc_id is None:
                raise KeyError(
                    "Processed examples must contain a document ID in "
                    "'id', 'doc_id', or 'document_id'. "
                    f"Found keys: {list(item.keys())}"
                )

            doc_id=str(doc_id)
            if doc_id not in self.doc_groups:
                self.doc_groups[doc_id]=[]

            self.doc_groups[doc_id].append(idx)

        self.doc_groups=list(self.doc_groups.values())

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.data[idx]

    def sample_indices(self,epoch,chunks_per_doc):
        if chunks_per_doc is None:
            return list(range(len(self.data)))

        selected=[]

        for group_idx,indices in enumerate(self.doc_groups):
            shuffled=indices.copy()
            random.Random(2026+group_idx).shuffle(shuffled)

            count=min(chunks_per_doc,len(shuffled))
            start=(epoch*chunks_per_doc)%len(shuffled)

            for j in range(count):
                selected.append(shuffled[(start+j)%len(shuffled)])

        return selected


class ListDataset(Dataset):
    def __init__(self,data):
        self.data=data

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.data[idx]

    def sample_indices(self,epoch,chunks_per_doc):
        return list(range(len(self.data)))


def collate_fn(batch):
    if len(batch)!=1:
        raise RuntimeError("This training setup requires BATCH_SIZE=1.")

    item=batch[0]
    source=item["source_ids"]
    target=item["target_ids"]

    if len(source)==0 or len(target)==0:
        raise RuntimeError("Encountered an empty source or target sequence.")

    if len(source)+len(target)-1>BLOCK_SIZE:
        raise RuntimeError(
            f"Combined input length {len(source)+len(target)-1} "
            f"exceeds BLOCK_SIZE={BLOCK_SIZE}."
        )

    seq=torch.tensor(source+target,dtype=torch.long)
    x=seq[:-1].unsqueeze(0)
    labels=torch.tensor(target,dtype=torch.long).unsqueeze(0)
    target_start=len(source)-1

    return x,labels,target_start


def make_smoke_dataset(dataset,count):
    data=[]

    for i in range(min(count,len(dataset))):
        item=dataset[i]
        source=list(item["source_ids"][:SMOKE_SOURCE_LEN])
        target=list(item["target_ids"])

        if len(target)>SMOKE_TARGET_LEN:
            target=target[:SMOKE_TARGET_LEN-1]+[0]
        else:
            target=target[:SMOKE_TARGET_LEN]

        data.append({
            "source_ids":source,
            "target_ids":target
        })

    return ListDataset(data)


# ================= ATTENTION BACKEND TEST =================

def test_efficient_attention(device):
    T=BLOCK_SIZE-2
    P=PREFIX_LEN
    H=12
    D=64
    W=512

    q=torch.randn(
        1,H,T,D,device=device,dtype=torch.float16,requires_grad=True
    )
    k=torch.randn(
        1,H,P+T,D,device=device,dtype=torch.float16,requires_grad=True
    )
    v=torch.randn(
        1,H,P+T,D,device=device,dtype=torch.float16,requires_grad=True
    )

    qpos=torch.arange(T,device=device)[:,None]
    kpos=torch.arange(T,device=device)[None,:]

    local_mask=(kpos<=qpos)&(kpos>=qpos-W+1)

    mask=torch.cat(
        [
            torch.ones(T,P,dtype=torch.bool,device=device),
            local_mask
        ],
        dim=1
    )[None,None,:,:]

    with sdpa_kernel(backends=[SDPBackend.EFFICIENT_ATTENTION]):
        out=F.scaled_dot_product_attention(
            q,k,v,attn_mask=mask,dropout_p=0.05
        )
        out.float().square().mean().backward()

    torch.cuda.synchronize(device)

    del q,k,v,mask,out
    torch.cuda.empty_cache()


# ================= VALIDATION =================

def validate(model,loader,device,rank,max_batches=None):
    model.eval()

    total_loss=torch.zeros((),device=device,dtype=torch.float64)
    total_tokens=torch.zeros((),device=device,dtype=torch.long)

    total_batches=len(loader) if max_batches is None else min(
        len(loader),max_batches
    )

    with torch.no_grad():
        progress=tqdm(
            loader,
            total=total_batches,
            desc="Validation",
            disable=(rank!=0),
            dynamic_ncols=True
        )

        for batch_idx,(x,y,target_start) in enumerate(progress):
            if max_batches is not None and batch_idx>=max_batches:
                break

            x=x.to(device,non_blocking=True)
            y=y.to(device,non_blocking=True)

            with autocast("cuda",dtype=torch.float16):
                logits=model(x,target_start)

            loss_sum=F.cross_entropy(
                logits.float().reshape(-1,VOCAB_SIZE),
                y.reshape(-1),
                reduction="sum"
            )

            total_loss+=loss_sum.detach().double()
            total_tokens+=y.numel()

    dist.all_reduce(total_loss,op=dist.ReduceOp.SUM)
    dist.all_reduce(total_tokens,op=dist.ReduceOp.SUM)

    model.train()

    return (total_loss/total_tokens.clamp_min(1)).item()


# ================= CHECKPOINTING =================

def save_latest(model,optimizer,scaler,scheduler,epoch,batch_idx,step,best_val):
    path=f"{CKPT_DIR}/latest_tmp.pt"

    torch.save(
        {
            "model":model.module.state_dict(),
            "optimizer":optimizer.state_dict(),
            "scaler":scaler.state_dict(),
            "scheduler":scheduler.state_dict(),
            "epoch":epoch,
            "batch_idx":batch_idx,
            "step":step,
            "best_val":best_val
        },
        path
    )

    os.replace(path,f"{CKPT_DIR}/latest.pt")


def check_ddp_sync(model):
    param=next(p for p in model.module.parameters() if p.requires_grad)

    value_min=param.detach().reshape(-1)[0].clone()
    value_max=value_min.clone()

    dist.all_reduce(value_min,op=dist.ReduceOp.MIN)
    dist.all_reduce(value_max,op=dist.ReduceOp.MAX)

    difference=(value_max-value_min).abs().item()

    if difference>1e-7:
        raise RuntimeError(f"DDP parameters differ across ranks: {difference}")

    return difference


# ================= MAIN =================

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--smoke-test",action="store_true")
    parser.add_argument("--resume",action="store_true")
    args=parser.parse_args()

    if args.smoke_test and args.resume:
        raise RuntimeError("Smoke test always starts with fresh prefix parameters.")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this DDP script.")

    local_rank=int(os.environ["LOCAL_RANK"])
    rank=int(os.environ["RANK"])
    world_size=int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    device=torch.device(f"cuda:{local_rank}")
    torch.set_num_threads(1)

    dist.init_process_group(backend="nccl")

    try:
        torch.manual_seed(42+rank)
        torch.backends.cuda.matmul.allow_tf32=True

        if rank==0:
            print("Testing full-length memory-efficient attention...")

        test_efficient_attention(device)
        dist.barrier()

        train_ds=SummaryDataset(TRAIN_PATH)
        valid_ds=SummaryDataset(VALID_PATH)

        if args.smoke_test:
            smoke_count=min(
                world_size*SMOKE_BATCHES,
                len(train_ds),
                len(valid_ds)
            )

            if smoke_count<world_size:
                raise RuntimeError("Not enough examples for the DDP smoke test.")

            train_ds=make_smoke_dataset(train_ds,smoke_count)
            valid_ds=make_smoke_dataset(valid_ds,smoke_count)

            grad_accum=1
            epochs=1
            num_workers=0
            max_train_batches=SMOKE_BATCHES
            max_valid_batches=SMOKE_BATCHES
        else:
            grad_accum=GRAD_ACCUM
            epochs=EPOCHS
            num_workers=NUM_WORKERS
            max_train_batches=None
            max_valid_batches=None

        train_indices=train_ds.sample_indices(
            0,None if args.smoke_test else TRAIN_CHUNKS_PER_DOC
        )
        valid_indices=valid_ds.sample_indices(
            0,None if args.smoke_test else VALID_CHUNKS_PER_DOC
        )

        train_epoch_size=len(train_indices)
        valid_eval_ds=Subset(valid_ds,valid_indices)

        valid_sampler=DistributedSampler(
            valid_eval_ds,
            num_replicas=world_size,
            rank=rank,
            shuffle=False,
            drop_last=False
        )

        valid_loader=DataLoader(
            valid_eval_ds,
            batch_size=BATCH_SIZE,
            sampler=valid_sampler,
            collate_fn=collate_fn,
            num_workers=num_workers,
            pin_memory=True,
            drop_last=False,
            persistent_workers=(num_workers>0)
        )

        # Load original Phase 1 M3 checkpoint.
        model=M3Model()

        pretrained=torch.load(
            PRETRAINED,
            map_location="cpu",
            weights_only=True
        )

        model.load_state_dict(pretrained,strict=True)

        # Add fresh trainable prefixes.
        model=install_prefixes(model)
        model.forward=types.MethodType(checkpointed_forward,model)
        model=model.to(device)

        trainable_params=[
            p for p in model.parameters() if p.requires_grad
        ]

        trainable_count=sum(p.numel() for p in trainable_params)
        expected_count=16*2*PREFIX_LEN*4*64

        if trainable_count!=expected_count:
            raise RuntimeError(
                f"Expected {expected_count} trainable prefix parameters, "
                f"got {trainable_count}."
            )

        optimizer=torch.optim.AdamW(
            trainable_params,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            fused=True
        )

        per_rank_train_batches=train_epoch_size//world_size
        effective_train_batches=(
            min(per_rank_train_batches,max_train_batches)
            if max_train_batches is not None else per_rank_train_batches
        )

        steps_per_epoch=math.ceil(effective_train_batches/grad_accum)
        total_steps=epochs*steps_per_epoch
        warmup_steps=max(1,int(0.03*total_steps))

        def lr_lambda(step):
            if step<warmup_steps:
                return (step+1)/warmup_steps

            progress=min(
                1.0,
                (step-warmup_steps)/max(1,total_steps-warmup_steps)
            )

            min_ratio=MIN_LR/LR

            return min_ratio+(1-min_ratio)*0.5*(
                1+math.cos(math.pi*progress)
            )

        scheduler=torch.optim.lr_scheduler.LambdaLR(
            optimizer,lr_lambda
        )

        scaler=GradScaler("cuda")

        model=DDP(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
            broadcast_buffers=False,
            find_unused_parameters=False
        )

        start_epoch=0
        start_batch=0
        global_step=0
        best_val=float("inf")

        resume_path=f"{CKPT_DIR}/latest.pt"

        if args.resume and os.path.exists(resume_path):
            ckpt=torch.load(
                resume_path,
                map_location="cpu",
                weights_only=False
            )

            model.module.load_state_dict(ckpt["model"],strict=True)
            optimizer.load_state_dict(ckpt["optimizer"])
            scaler.load_state_dict(ckpt["scaler"])
            scheduler.load_state_dict(ckpt["scheduler"])

            start_epoch=ckpt["epoch"]
            start_batch=ckpt["batch_idx"]
            global_step=ckpt["step"]
            best_val=ckpt["best_val"]

            if rank==0:
                print("Resumed from epoch:",start_epoch+1)
                print("Next batch:",start_batch)
                print("Optimizer steps:",global_step)

        elif args.resume and rank==0:
            print("No latest checkpoint found; starting fresh.")

        if rank==0:
            print("Training chunks available:",len(train_ds))
            print("Training chunks selected per epoch:",train_epoch_size)
            print("Training documents:",len(train_ds.doc_groups) if isinstance(train_ds,SummaryDataset) else len(train_ds))
            print("Validation chunks selected:",len(valid_eval_ds))
            print("GPUs:",world_size)
            print("Trainable parameters:",trainable_count)
            print("Steps per epoch:",steps_per_epoch)
            print("Total planned optimizer steps:",total_steps)
            print("Smoke test:",args.smoke_test)

        for epoch in range(start_epoch,epochs):
            epoch_indices=train_ds.sample_indices(
                epoch,None if args.smoke_test else TRAIN_CHUNKS_PER_DOC
            )
            epoch_ds=Subset(train_ds,epoch_indices)

            train_sampler=DistributedSampler(
                epoch_ds,
                num_replicas=world_size,
                rank=rank,
                shuffle=True,
                drop_last=True
            )

            train_loader=DataLoader(
                epoch_ds,
                batch_size=BATCH_SIZE,
                sampler=train_sampler,
                collate_fn=collate_fn,
                num_workers=num_workers,
                pin_memory=True,
                drop_last=True,
                persistent_workers=(num_workers>0)
            )

            train_sampler.set_epoch(epoch)
            model.train()

            initial_batch=start_batch if epoch==start_epoch else 0
            epoch_batches=min(len(train_loader),effective_train_batches)

            if epoch_batches!=effective_train_batches:
                raise RuntimeError(
                    f"Epoch batch count changed: expected {effective_train_batches}, "
                    f"got {epoch_batches}."
                )

            progress=tqdm(
                total=epoch_batches,
                initial=initial_batch,
                desc=f"M3 Prefix Epoch {epoch+1}/{epochs}",
                disable=(rank!=0),
                dynamic_ncols=True
            )

            optimizer.zero_grad(set_to_none=True)
            running_loss=0.0
            running_count=0
            last_grad_norm=0.0

            for batch_idx,(x,y,target_start) in enumerate(train_loader):
                if batch_idx>=epoch_batches:
                    break

                if batch_idx<initial_batch:
                    continue

                group_start=(batch_idx//grad_accum)*grad_accum
                group_size=min(grad_accum,epoch_batches-group_start)
                sync_now=(batch_idx+1==group_start+group_size)

                x=x.to(device,non_blocking=True)
                y=y.to(device,non_blocking=True)

                sync_context=nullcontext() if sync_now else model.no_sync()

                with sync_context:
                    with autocast("cuda",dtype=torch.float16):
                        logits=model(x,target_start)

                        loss=F.cross_entropy(
                            logits.float().reshape(-1,VOCAB_SIZE),
                            y.reshape(-1)
                        )/group_size

                    if not torch.isfinite(loss):
                        raise RuntimeError(
                            f"Non-finite loss at epoch {epoch+1}, batch {batch_idx}."
                        )

                    scaler.scale(loss).backward()

                running_loss+=loss.item()*group_size
                running_count+=1

                if sync_now:
                    scaler.unscale_(optimizer)

                    grad_norm=torch.nn.utils.clip_grad_norm_(
                        trainable_params,GRAD_CLIP
                    )

                    last_grad_norm=float(grad_norm.detach().float().item())

                    if not math.isfinite(last_grad_norm):
                        raise RuntimeError("Non-finite prefix gradient norm.")

                    if args.smoke_test and last_grad_norm<=0:
                        raise RuntimeError("Smoke test produced zero prefix gradients.")

                    scaler.step(optimizer)
                    scaler.update()

                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()
                    global_step+=1

                    if global_step%SAVE_EVERY==0 and not args.smoke_test:
                        if rank==0:
                            save_latest(
                                model,optimizer,scaler,scheduler,
                                epoch,batch_idx+1,global_step,best_val
                            )

                        dist.barrier()

                if rank==0:
                    progress.update(1)

                    if sync_now:
                        progress.set_postfix({
                            "loss":f"{running_loss/max(1,running_count):.4f}",
                            "grad":f"{last_grad_norm:.3f}",
                            "lr":f"{optimizer.param_groups[0]['lr']:.2e}",
                            "step":global_step
                        })

                        running_loss=0.0
                        running_count=0

            progress.close()
            start_batch=0

            if args.smoke_test:
                sync_difference=check_ddp_sync(model)

                if rank==0:
                    print("Smoke training gradient norm:",last_grad_norm)
                    print("DDP parameter max difference:",sync_difference)

            val_loss=validate(
                model.module,
                valid_loader,
                device,
                rank,
                max_batches=max_valid_batches
            )

            if args.smoke_test:
                if rank==0:
                    torch.save(
                        {
                            "model":model.module.state_dict(),
                            "validation_loss":val_loss,
                            "trainable_parameters":trainable_count
                        },
                        SMOKE_CKPT
                    )

                    check=torch.load(
                        SMOKE_CKPT,
                        map_location="cpu",
                        weights_only=False
                    )

                    if "model" not in check or not math.isfinite(check["validation_loss"]):
                        raise RuntimeError("Smoke checkpoint verification failed.")

                dist.barrier()

                if not os.path.exists(SMOKE_CKPT):
                    raise RuntimeError("Smoke checkpoint was not created.")

                if rank==0:
                    print("Smoke validation loss:",val_loss)
                    print("Smoke checkpoint saved and reloaded:",SMOKE_CKPT)
                    print("DDP smoke test PASSED. No full-training checkpoint was modified.")

                break

            if rank==0:
                print(f"\nEpoch {epoch+1} validation loss: {val_loss:.5f}")

                if val_loss<best_val:
                    best_val=val_loss
                    torch.save(
                        model.module.state_dict(),
                        f"{CKPT_DIR}/best.pt"
                    )
                    print("Saved new best checkpoint.")

                save_latest(
                    model,optimizer,scaler,scheduler,
                    epoch+1,0,global_step,best_val
                )

            dist.barrier()

        if rank==0 and not args.smoke_test:
            print("\nM3 Phase 6 Prefix Tuning complete.")
            print("Best validation loss:",best_val)
            print("Checkpoints:",CKPT_DIR)

    finally:
        if dist.is_initialized():
            dist.destroy_process_group()


if __name__=="__main__":
    main()

Overwriting /kaggle/working/train_m3_phase6.py


In [28]:
!OMP_NUM_THREADS=1 torchrun --standalone --nproc_per_node=2 /kaggle/working/train_m3_phase6.py

Testing full-length memory-efficient attention...
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1103 09:00:08.095130792 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
Training chunks available: 66782
Training chunks selected per epoch: 12604
Training documents: 6327
Validation chunks selected: 1401
GPUs: 2
Trainable parameters: 262144
Steps per epoch: 788
Total planned optimizer steps: 2364
Smoke test: False
M3 Prefix Epoch 1/3:  63%|▋| 3999/6302 [20:43<11:47,  3.25it/s, loss=1.7130, gra/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device

In [31]:
import types
import torch
import torch.nn.functional as F

def prefix_attention_forward(attn,x):

    B,T,C=x.shape
    P=PREFIX_LEN

    q=attn.q_proj(x)
    k=attn.k_proj(x)
    v=attn.v_proj(x)

    q=q.view(B,T,attn.n_heads,attn.head_dim).transpose(1,2)
    k=k.view(B,T,attn.n_kv_heads,attn.head_dim).transpose(1,2)
    v=v.view(B,T,attn.n_kv_heads,attn.head_dim).transpose(1,2)

    positions=torch.arange(T,device=x.device)+P

    def apply_rope(z,pos):
        cos=attn.prefix_rope_cos.index_select(0,pos).to(z.dtype)[None,None,:,:]
        sin=attn.prefix_rope_sin.index_select(0,pos).to(z.dtype)[None,None,:,:]

        z1=z[...,::2]
        z2=z[...,1::2]

        return torch.stack(
            [z1*cos-z2*sin,z1*sin+z2*cos],
            dim=-1
        ).flatten(-2)

    q=apply_rope(q,positions)
    k=apply_rope(k,positions)

    prefix_positions=torch.arange(P,device=x.device)

    prefix_k=apply_rope(attn.prefix_k.to(k.dtype),prefix_positions)
    prefix_v=attn.prefix_v.to(v.dtype)

    k=torch.cat([prefix_k.expand(B,-1,-1,-1),k],dim=-2)
    v=torch.cat([prefix_v.expand(B,-1,-1,-1),v],dim=-2)

    mask=attn.prefix_mask[:,:,:T,:P+T]

    repeat_factor=attn.n_heads//attn.n_kv_heads
    k=k.repeat_interleave(repeat_factor,dim=1)
    v=v.repeat_interleave(repeat_factor,dim=1)

    y=F.scaled_dot_product_attention(
        q,k,v,
        attn_mask=mask,
        dropout_p=0.0
    )

    y=y.transpose(1,2).contiguous().view(
        B,T,attn.n_heads*attn.head_dim
    )

    return attn.o_proj(y)


# Install the trained Prefix Tuning attention in every layer.
for block in model.blocks:
    block.attn.forward=types.MethodType(
        prefix_attention_forward,
        block.attn
    )

model.eval()

print("Prefix attention installed in all layers.")


# ============================================================
# RETEST KV CACHE
# ============================================================

test_ids=torch.randint(
    0,tokenizer.vocab_size,
    (1,65),
    device=DEVICE
)

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(DEVICE.type=="cuda")
    ):

        # Normal forward now includes the learned prefixes.
        normal_logits=model(test_ids)

        # Cached prefill: first 64 tokens.
        first_logits,cache=forward_cached(
            model,test_ids[:,:64]
        )

        # Cached decoding: token at position 64.
        next_logits,cache=forward_cached(
            model,
            test_ids[:,64:65],
            cache,
            start_pos=64
        )

    expected=normal_logits[:,-1,:]
    actual=next_logits[:,-1,:]

    max_diff=(expected-actual).abs().max().item()
    mean_diff=(expected-actual).abs().mean().item()

print("Maximum logit difference:",max_diff)
print("Mean logit difference:",mean_diff)

assert torch.allclose(
    expected,actual,
    rtol=0.03,
    atol=0.03
), "KV cache still does not match normal Prefix Tuning forward."

print("KV cache equivalence check PASSED.")

Prefix attention installed in all layers.


NameError: name 'tokenizer' is not defined

In [32]:
import os
import json
import zipfile
import torch
from tqdm.auto import tqdm
from collections import OrderedDict

# ================= CONFIGURATION =================

OUTPUT_DIR = "/kaggle/working"
JSON_PATH = os.path.join(OUTPUT_DIR, "submission_m3.json")
ZIP_PATH = os.path.join(OUTPUT_DIR, "submission_m3.zip")

MAX_NEW_TOKENS = 500
EOS_ID = 0
BLOCK_SIZE = 2048

# Use the tokenizer already loaded in the notebook.
if "tok" in globals():
    tokenizer = tok

DEVICE = torch.device(DEVICE)
model.eval()

assert len(test_chunks) > 0
assert callable(forward_cached)
assert hasattr(model, "forward_cached") or callable(forward_cached)

# ================= TOKEN DECODING =================

def decode_tokens(ids):
    output_bytes = bytearray()

    for token_id in ids:
        token_id = int(token_id)

        if token_id == tokenizer.eos_id:
            continue

        output_bytes.extend(tokenizer.itos[token_id])

    return output_bytes.decode("utf-8", errors="replace")


# ================= SUMMARY GENERATION =================

@torch.no_grad()
def generate_summary(source_ids):
    source_ids = list(source_ids)

    if len(source_ids) == 0:
        return ""

    if len(source_ids) >= BLOCK_SIZE:
        raise ValueError(
            f"Source length {len(source_ids)} reaches BLOCK_SIZE={BLOCK_SIZE}."
        )

    max_tokens = min(
        MAX_NEW_TOKENS,
        BLOCK_SIZE - len(source_ids)
    )

    x = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=DEVICE
    )

    generated = []

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(DEVICE.type == "cuda")
    ):
        # Process the complete source once.
        logits, cache = forward_cached(model, x)

        for step in range(max_tokens):
            next_token = logits[:, -1, :].argmax(dim=-1).item()

            if next_token == EOS_ID:
                break

            generated.append(next_token)

            if step + 1 == max_tokens:
                break

            # Process only the newly generated token.
            next_input = torch.tensor(
                [[next_token]],
                dtype=torch.long,
                device=DEVICE
            )

            logits, cache = forward_cached(
                model,
                next_input,
                cache,
                start_pos=len(source_ids) + step
            )

    return decode_tokens(generated).strip()


# ================= GENERATE ALL CHUNK SUMMARIES =================

chunk_results = []

for i, item in enumerate(tqdm(test_chunks, desc="Generating M3 summaries")):
    summary = generate_summary(item["source_ids"])

    chunk_results.append({
        "id": str(item["id"]),
        "chunk_idx": int(item["chunk_idx"]),
        "summary": summary
    })

    # Save intermediate progress every 50 chunks.
    if (i + 1) % 50 == 0:
        with open(
            os.path.join(OUTPUT_DIR, "m3_generation_progress.json"),
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(chunk_results, f, ensure_ascii=False)

print("\nGenerated summaries for", len(chunk_results), "chunks.")


# ================= COMBINE CHUNKS BY DOCUMENT =================

grouped = OrderedDict()

for item in chunk_results:
    doc_id = item["id"]

    if doc_id not in grouped:
        grouped[doc_id] = []

    grouped[doc_id].append(item)

submission = []

# Preserve the original test document order if test_data is available.
if "test_data" in globals():
    document_ids = [str(item["id"]) for item in test_data]
else:
    document_ids = list(grouped.keys())

for doc_id in document_ids:
    if doc_id not in grouped:
        raise RuntimeError(f"No generated chunks found for {doc_id}")

    chunks = sorted(
        grouped[doc_id],
        key=lambda item: item["chunk_idx"]
    )

    summary_parts = [
        item["summary"]
        for item in chunks
        if item["summary"].strip()
    ]

    final_summary = " ".join(summary_parts).strip()

    submission.append({
        "id": doc_id,
        "summary": final_summary
    })


# ================= SAVE JSON =================

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print("\nSubmission documents:", len(submission))
print("Empty summaries:", sum(1 for x in submission if not x["summary"]))
print("JSON saved:", JSON_PATH)


# ================= CREATE ZIP =================

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(JSON_PATH, arcname="submission.json")

# Verify ZIP contents and JSON structure.
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    assert z.namelist() == ["submission.json"]

    check = json.loads(
        z.read("submission.json").decode("utf-8")
    )

    assert len(check) == len(submission)
    assert all(
        set(item.keys()) == {"id", "summary"}
        for item in check
    )

print("\nM3 submission ZIP created and verified.")
print("ZIP:", ZIP_PATH)
print("Documents:", len(check))
print("ZIP size:", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2), "MB")

NameError: name 'DEVICE' is not defined

In [36]:

import types
import torch
import torch.nn.functional as F

PREFIX_LEN=32
WINDOW_SIZE=512

def rotate_m3(x,cos,sin):
    x1=x[...,::2]
    x2=x[...,1::2]
    return torch.stack(
        [x1*cos-x2*sin,x1*sin+x2*cos],
        dim=-1
    ).flatten(-2)

def cached_attention(attn,x,past_kv,start_pos):
    B,T,C=x.shape
    P=attn.prefix_len

    q=attn.q_proj(x)
    k=attn.k_proj(x)
    v=attn.v_proj(x)

    q=q.view(B,T,attn.n_heads,attn.head_dim).transpose(1,2)
    k=k.view(B,T,attn.n_kv_heads,attn.head_dim).transpose(1,2)
    v=v.view(B,T,attn.n_kv_heads,attn.head_dim).transpose(1,2)

    positions=torch.arange(
        start_pos+P,start_pos+P+T,device=x.device
    )

    cos=attn.prefix_rope_cos.index_select(0,positions).to(q.dtype)[None,None,:,:]
    sin=attn.prefix_rope_sin.index_select(0,positions).to(q.dtype)[None,None,:,:]

    q=rotate_m3(q,cos,sin)
    k=rotate_m3(k,cos,sin)

    if past_kv is None:
        prefix_positions=torch.arange(P,device=x.device)
        prefix_cos=attn.prefix_rope_cos.index_select(
            0,prefix_positions
        ).to(k.dtype)[None,None,:,:]
        prefix_sin=attn.prefix_rope_sin.index_select(
            0,prefix_positions
        ).to(k.dtype)[None,None,:,:]

        prefix_k=rotate_m3(attn.prefix_k.to(k.dtype),prefix_cos,prefix_sin)
        prefix_v=attn.prefix_v.to(v.dtype)

        content_k=k
        content_v=v
    else:
        past_k,past_v=past_kv
        content_k=torch.cat([past_k,k],dim=2)
        content_v=torch.cat([past_v,v],dim=2)

        # Keep only the previous 511 content tokens.
        content_k=content_k[:,:,-(WINDOW_SIZE-1):,:]
        content_v=content_v[:,:,-(WINDOW_SIZE-1):,:]

        prefix_k=None
        prefix_v=None

    if past_kv is None:
        all_k=torch.cat([prefix_k.expand(B,-1,-1,-1),content_k],dim=2)
        all_v=torch.cat([prefix_v.expand(B,-1,-1,-1),content_v],dim=2)
    else:
        prefix_positions=torch.arange(P,device=x.device)
        prefix_cos=attn.prefix_rope_cos.index_select(
            0,prefix_positions
        ).to(k.dtype)[None,None,:,:]
        prefix_sin=attn.prefix_rope_sin.index_select(
            0,prefix_positions
        ).to(k.dtype)[None,None,:,:]

        prefix_k=rotate_m3(attn.prefix_k.to(k.dtype),prefix_cos,prefix_sin)
        prefix_v=attn.prefix_v.to(v.dtype)

        all_k=torch.cat([prefix_k.expand(B,-1,-1,-1),content_k],dim=2)
        all_v=torch.cat([prefix_v.expand(B,-1,-1,-1),content_v],dim=2)

    repeat_factor=attn.n_heads//attn.n_kv_heads
    all_k=all_k.repeat_interleave(repeat_factor,dim=1)
    all_v=all_v.repeat_interleave(repeat_factor,dim=1)

    if T==1:
        mask=None
    else:
        # Used for prefill only: prefix is always visible,
        # content follows causal sliding-window attention.
        query_pos=torch.arange(start_pos,start_pos+T,device=x.device)[:,None]
        key_pos=torch.arange(start_pos+T,device=x.device)[None,:]
        local=(key_pos<=query_pos)&(key_pos>=query_pos-WINDOW_SIZE+1)
        mask=torch.cat(
            [torch.ones(T,P,dtype=torch.bool,device=x.device),local],
            dim=1
        )[None,None,:,:]

    y=F.scaled_dot_product_attention(
        q,all_k,all_v,
        attn_mask=mask,
        dropout_p=0.0
    )

    y=y.transpose(1,2).contiguous().view(B,T,attn.n_heads*attn.head_dim)
    return attn.o_proj(y),(content_k,content_v)


def block_cached(block,x,past_kv,start_pos):
    attn_out,new_kv=cached_attention(
        block.attn,block.ln1(x),past_kv,start_pos
    )
    x=x+block.dropout(attn_out)
    x=x+block.dropout(block.ffn(block.ln2(x)))
    return x,new_kv


def m3_forward_cached(self,idx,past_kvs=None,start_pos=0):
    x=self.tok_emb(idx)
    new_kvs=[]

    if past_kvs is None:
        past_kvs=[None]*len(self.blocks)

    for i,block in enumerate(self.blocks):
        x,new_kv=block_cached(
            block,x,past_kvs[i],start_pos
        )
        new_kvs.append(new_kv)

    x=self.ln_f(x)
    logits=F.linear(x[:,-1:,:],self.tok_emb.weight)

    return logits,new_kvs


model.forward_cached=types.MethodType(m3_forward_cached,model)
model.eval()

print("Optimized KV cache installed.")


Optimized KV cache installed.


In [20]:

import torch

test_item=test_chunks[0]
test_ids=test_item["source_ids"]

x=torch.tensor([test_ids],dtype=torch.long,device=next(model.parameters()).device)

with torch.no_grad():
    with torch.autocast("cuda",dtype=torch.float16):
        full_logits=model(x)[:,-1:,:]
        cached_logits,_=model.forward_cached(x)

diff=(full_logits.float()-cached_logits.float()).abs()

print("Maximum difference:",diff.max().item())
print("Mean difference:",diff.mean().item())


Maximum difference: 0.00390625
Mean difference: 3.816228399955435e-06


In [ ]:

import os
import json
import time
import torch
from tqdm.auto import tqdm
from collections import defaultdict

device=next(model.parameters()).device
model.eval()

EOS_ID=0
MAX_NEW_TOKENS=500
SAVE_EVERY=25

PROGRESS_PATH="/kaggle/working/m3_generation_progress.json"
OUTPUT_PATH="/kaggle/working/submission_m3.json"
ZIP_PATH="/kaggle/working/submission_m3.zip"

def decode_tokens(ids):
    output_bytes=bytearray()

    for token_id in ids:
        token_id=int(token_id)

        if token_id==EOS_ID:
            continue

        output_bytes.extend(tokenizer.itos[token_id])

    return output_bytes.decode("utf-8",errors="replace")


@torch.no_grad()
def generate_summary(source_ids):
    x=torch.tensor([source_ids],dtype=torch.long,device=device)
    generated=[]

    max_tokens=min(MAX_NEW_TOKENS,2048-len(source_ids))

    with torch.autocast("cuda",dtype=torch.float16):
        logits,past=model.forward_cached(x)
        token=logits[0,-1].argmax().item()

        for step in range(max_tokens):
            if token==EOS_ID:
                break

            generated.append(token)

            next_x=torch.tensor([[token]],dtype=torch.long,device=device)

            logits,past=model.forward_cached(
                next_x,
                past,
                start_pos=len(source_ids)+step
            )

            token=logits[0,-1].argmax().item()

    return decode_tokens(generated).strip()


# Resume from saved progress if available.
if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH,"r",encoding="utf-8") as f:
        progress=json.load(f)

    predictions=progress["predictions"]
    completed=set(progress["completed"])

    print("Resuming:",len(completed),"chunks already completed.")
else:
    predictions={}
    completed=set()

start_time=time.time()
lengths=[]

for i,item in enumerate(tqdm(test_chunks,desc="Generating M3 summaries")):
    chunk_key=str(i)

    if chunk_key in completed:
        continue

    summary=generate_summary(item["source_ids"])

    predictions[chunk_key]={
        "id":str(item["id"]),
        "chunk_idx":int(item["chunk_idx"]),
        "summary":summary
    }

    completed.add(chunk_key)
    lengths.append(len(summary))

    if len(completed)%SAVE_EVERY==0:
        with open(PROGRESS_PATH,"w",encoding="utf-8") as f:
            json.dump(
                {
                    "predictions":predictions,
                    "completed":list(completed)
                },
                f,
                ensure_ascii=False
            )

        elapsed=time.time()-start_time
        print(
            "\nCompleted:",len(completed),
            "| Elapsed:",round(elapsed/60,1),"min"
        )

# Final progress save.
with open(PROGRESS_PATH,"w",encoding="utf-8") as f:
    json.dump(
        {
            "predictions":predictions,
            "completed":list(completed)
        },
        f,
        ensure_ascii=False
    )

# Combine chunk summaries in their original order.
grouped=defaultdict(list)

for value in predictions.values():
    grouped[value["id"]].append(value)

submission=[]

for row in test_data:
    doc_id=str(row["id"])
    chunks=sorted(grouped[doc_id],key=lambda x:x["chunk_idx"])

    combined=" ".join(
        x["summary"] for x in chunks if x["summary"]
    ).strip()

    submission.append({
        "id":row["id"],
        "summary":combined
    })

with open(OUTPUT_PATH,"w",encoding="utf-8") as f:
    json.dump(submission,f,ensure_ascii=False,indent=2)

import zipfile

with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(OUTPUT_PATH,arcname="submission.json")

print("\nGeneration complete.")
print("Chunks:",len(completed))
print("Documents:",len(submission))
print("Elapsed minutes:",round((time.time()-start_time)/60,2))
print("Submission:",OUTPUT_PATH)
print("ZIP:",ZIP_PATH)


In [49]:
def decode_tokens(ids):
    output_bytes=bytearray()

    for token_id in ids:
        token_id=int(token_id)

        if token_id==EOS_ID:
            continue

        output_bytes.extend(tokenizer.itos[token_id])

    return output_bytes.decode("utf-8",errors="replace")

In [ ]:

import os
import json
import time
import threading
import torch
from tqdm.auto import tqdm

# Both models must be on their respective GPUs.
model=model.eval()
model2=model2.eval()

device0=torch.device("cuda:0")
device1=torch.device("cuda:1")

assert next(model.parameters()).device==device0
assert next(model2.parameters()).device==device1

PROGRESS_PATH="/kaggle/working/m3_generation_progress.json"
SAVE_EVERY=25
MAX_NEW_TOKENS=500
EOS_ID=0

# Load previously saved progress.
if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH,"r",encoding="utf-8") as f:
        progress=json.load(f)
    predictions=progress["predictions"]
    completed=set(progress["completed"])
else:
    predictions={}
    completed=set()

print("Already completed:",len(completed))


def generate_on_gpu(m,device,source_ids):
    x=torch.tensor([source_ids],dtype=torch.long,device=device)
    generated=[]

    max_tokens=min(MAX_NEW_TOKENS,2048-len(source_ids))

    with torch.no_grad():
        with torch.autocast("cuda",dtype=torch.float16):
            logits,past=m.forward_cached(x)
            token=logits[0,-1].argmax().item()

            for step in range(max_tokens):
                if token==EOS_ID:
                    break

                generated.append(token)

                next_x=torch.tensor([[token]],dtype=torch.long,device=device)

                logits,past=m.forward_cached(
                    next_x,
                    past,
                    start_pos=len(source_ids)+step
                )

                token=logits[0,-1].argmax().item()

    return decode_tokens(generated).strip()


# Divide remaining chunks between GPUs.
remaining=[]
for i in range(len(test_chunks)):
    if str(i) not in completed:
        remaining.append(i)

indices0=remaining[::2]
indices1=remaining[1::2]

print("GPU 0 chunks:",len(indices0))
print("GPU 1 chunks:",len(indices1))


# Shared state and synchronization.
lock=threading.Lock()
bar=tqdm(total=len(remaining),desc="Two-GPU generation")
errors=[]


def save_progress():
    with open(PROGRESS_PATH,"w",encoding="utf-8") as f:
        json.dump(
            {
                "predictions":predictions,
                "completed":list(completed)
            },
            f,
            ensure_ascii=False
        )


def worker(m,device,indices):
    try:
        for i in indices:
            item=test_chunks[i]

            summary=generate_on_gpu(
                m,
                device,
                item["source_ids"]
            )

            with lock:
                predictions[str(i)]={
                    "id":str(item["id"]),
                    "chunk_idx":int(item["chunk_idx"]),
                    "summary":summary
                }

                completed.add(str(i))
                bar.update(1)

                if len(completed)%SAVE_EVERY==0:
                    save_progress()

    except Exception as e:
        with lock:
            errors.append(str(e))


start_time=time.time()

thread0=threading.Thread(
    target=worker,
    args=(model,device0,indices0)
)

thread1=threading.Thread(
    target=worker,
    args=(model2,device1,indices1)
)

thread0.start()
thread1.start()

thread0.join()
thread1.join()

bar.close()

# Save even if a worker encountered an error.
with lock:
    save_progress()

print("Completed chunks:",len(completed))
print("Elapsed minutes:",round((time.time()-start_time)/60,2))
print("Errors:",errors)


Already completed: 0
GPU 0 chunks: 576
GPU 1 chunks: 576


Two-GPU generation:   0%|          | 0/1152 [00:00<?, ?it/s]

In [ ]:

import os
import json
import time
import threading
import torch
from tqdm.auto import tqdm

# Both models must be on their respective GPUs.
model=model.eval()
model2=model2.eval()

device0=torch.device("cuda:0")
device1=torch.device("cuda:1")

assert next(model.parameters()).device==device0
assert next(model2.parameters()).device==device1

PROGRESS_PATH="/kaggle/working/m3_generation_progress.json"
SAVE_EVERY=25
MAX_NEW_TOKENS=500
EOS_ID=0

# Load previously saved progress.
if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH,"r",encoding="utf-8") as f:
        progress=json.load(f)
    predictions=progress["predictions"]
    completed=set(progress["completed"])
else:
    predictions={}
    completed=set()

print("Already completed:",len(completed))


def generate_on_gpu(m,device,source_ids):
    x=torch.tensor([source_ids],dtype=torch.long,device=device)
    generated=[]

    max_tokens=min(MAX_NEW_TOKENS,2048-len(source_ids))

    with torch.no_grad():
        with torch.autocast("cuda",dtype=torch.float16):
            logits,past=m.forward_cached(x)
            token=logits[0,-1].argmax().item()

            for step in range(max_tokens):
                if token==EOS_ID:
                    break

                generated.append(token)

                next_x=torch.tensor([[token]],dtype=torch.long,device=device)

                logits,past=m.forward_cached(
                    next_x,
                    past,
                    start_pos=len(source_ids)+step
                )

                token=logits[0,-1].argmax().item()

    return decode_tokens(generated).strip()


# Divide remaining chunks between GPUs.
remaining=[]
for i in range(len(test_chunks)):
    if str(i) not in completed:
        remaining.append(i)

indices0=remaining[::2]
indices1=remaining[1::2]

print("GPU 0 chunks:",len(indices0))
print("GPU 1 chunks:",len(indices1))


# Shared state and synchronization.
lock=threading.Lock()
bar=tqdm(total=len(remaining),desc="Two-GPU generation")
errors=[]


def save_progress():
    with open(PROGRESS_PATH,"w",encoding="utf-8") as f:
        json.dump(
            {
                "predictions":predictions,
                "completed":list(completed)
            },
            f,
            ensure_ascii=False
        )


def worker(m,device,indices):
    try:
        for i in indices:
            item=test_chunks[i]

            summary=generate_on_gpu(
                m,
                device,
                item["source_ids"]
            )

            with lock:
                predictions[str(i)]={
                    "id":str(item["id"]),
                    "chunk_idx":int(item["chunk_idx"]),
                    "summary":summary
                }

                completed.add(str(i))
                bar.update(1)

                if len(completed)%SAVE_EVERY==0:
                    save_progress()

    except Exception as e:
        with lock:
            errors.append(str(e))


start_time=time.time()

thread0=threading.Thread(
    target=worker,
    args=(model,device0,indices0)
)

thread1=threading.Thread(
    target=worker,
    args=(model2,device1,indices1)
)

thread0.start()
thread1.start()

thread0.join()
thread1.join()

bar.close()

# Save even if a worker encountered an error.
with lock:
    save_progress()

print("Completed chunks:",len(completed))
print("Elapsed minutes:",round((time.time()-start_time)/60,2))
print("Errors:",errors)


Already completed: 325
GPU 0 chunks: 414
GPU 1 chunks: 413


Two-GPU generation:   0%|          | 0/827 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [16]:
import torch
import copy

PREFIX_LEN=32

def restore_prefix_buffers(m):
    for block in m.blocks:
        attn=block.attn

        assert hasattr(attn,"prefix_k")
        assert hasattr(attn,"prefix_v")

        device=attn.prefix_k.device
        dtype=attn.prefix_k.dtype

        # First 32 RoPE positions are used by the learned prefix.
        attn.prefix_rope_cos=attn.rope.cos[:PREFIX_LEN].to(
            device=device,dtype=dtype
        )[None,None,:,:]

        attn.prefix_rope_sin=attn.rope.sin[:PREFIX_LEN].to(
            device=device,dtype=dtype
        )[None,None,:,:]

        attn.prefix_mask=torch.ones(
            (1,1,1,PREFIX_LEN),
            dtype=torch.bool,
            device=device
        )

        attn.prefix_len=PREFIX_LEN

    return m


model=restore_prefix_buffers(model)

print("GPU 0 prefix buffers restored.")

AssertionError: 

In [42]:
model2=copy.deepcopy(model).to("cuda:1")
model=model.to("cuda:0")

model.eval()
model2.eval()

print("GPU 0:",next(model.parameters()).device)
print("GPU 1:",next(model2.parameters()).device)

GPU 0: cuda:0
GPU 1: cuda:1


In [47]:
import torch

def check_cached_model(m,device,source_ids):
    x=torch.tensor([source_ids],dtype=torch.long,device=device)

    with torch.no_grad():
        with torch.autocast("cuda",dtype=torch.float16):
            normal_logits=m(x)[:,-1,:]
            cached_logits,_=m.forward_cached(x)

    diff=(normal_logits-cached_logits[:, -1, :]).abs()

    return diff.max().item(),diff.mean().item()


test_ids=test_chunks[0]["source_ids"]

for name,m,device in [
    ("GPU 0",model,"cuda:0"),
    ("GPU 1",model2,"cuda:1")
]:
    max_diff,mean_diff=check_cached_model(m,device,test_ids)

    print(name)
    print("Maximum difference:",max_diff)
    print("Mean difference:",mean_diff)

RuntimeError: The expanded size of the tensor (1056) must match the existing size (32) at non-singleton dimension 3.  Target sizes: [1, 12, 1024, 1056].  Tensor sizes: [1, 1, 1, 32]

In [44]:
import torch

def fix_prefix_rope(m,device):
    m=m.to(device)

    for block in m.blocks:
        attn=block.attn

        # Use the complete RoPE tables.
        attn.prefix_rope_cos=attn.rope.cos
        attn.prefix_rope_sin=attn.rope.sin

        attn.prefix_len=32

        attn.prefix_mask=torch.ones(
            (1,1,1,32),
            dtype=torch.bool,
            device=device
        )

        assert attn.prefix_rope_cos.device==device
        assert attn.prefix_rope_sin.device==device
        assert attn.prefix_rope_cos.dim()==2

    return m


model=fix_prefix_rope(model,torch.device("cuda:0"))
model2=fix_prefix_rope(model2,torch.device("cuda:1"))

model.eval()
model2.eval()

print("GPU 0:",model.blocks[0].attn.prefix_rope_cos.shape,
      model.blocks[0].attn.prefix_rope_cos.device)

print("GPU 1:",model2.blocks[0].attn.prefix_rope_cos.shape,
      model2.blocks[0].attn.prefix_rope_cos.device)

GPU 0: torch.Size([2048, 32]) cuda:0
GPU 1: torch.Size([2048, 32]) cuda:1


In [39]:
def generate_tokens_on_model(m,device,source_ids,max_new_tokens=64):
    m.eval()
    ids=list(source_ids)
    generated=[]

    with torch.no_grad():
        with torch.autocast("cuda",dtype=torch.float16):
            x=torch.tensor([ids],dtype=torch.long,device=device)
            logits,past_kvs=m.forward_cached(x,start_pos=0)

            for step in range(max_new_tokens):
                token=logits[0,-1].argmax().item()

                if token==0:
                    break

                generated.append(token)
                ids.append(token)

                x=torch.tensor([[token]],dtype=torch.long,device=device)

                logits,past_kvs=m.forward_cached(
                    x,
                    past_kvs=past_kvs,
                    start_pos=len(ids)-1
                )

    return generated

In [45]:
tokens_cached=generate_tokens_on_model(
    model,"cuda:0",test_ids,64
)

tokens_full=generate_full_tokens(
    model,"cuda:0",test_ids,64
)

print("Cached length:",len(tokens_cached))
print("Full length:",len(tokens_full))
print("Identical tokens:",tokens_cached==tokens_full)

if tokens_cached!=tokens_full:
    for i in range(min(len(tokens_cached),len(tokens_full))):
        if tokens_cached[i]!=tokens_full[i]:
            print("First mismatch at token:",i)
            print("Cached:",tokens_cached[i])
            print("Full:",tokens_full[i])
            break

NameError: name 'generate_full_tokens' is not defined

In [46]:
print("GPU 0:",model.forward_cached is not None)
print("GPU 1:",model2.forward_cached is not None)

print("GPU 0 device:",next(model.parameters()).device)
print("GPU 1 device:",next(model2.parameters()).device)

print("Completed chunks:",len(progress["completed"]))

GPU 0: True
GPU 1: True
GPU 0 device: cuda:0
GPU 1 device: cuda:1


NameError: name 'progress' is not defined